# 한국로텍 데이터 분석

## 실행 전에 준비할 자료
이 공개용 노트북에는 회사 측정값과 이미지 묶음이 들어 있지 않습니다.
담당자가 별도로 제공한 `raw`, `csv`, `prepared` 폴더를 노트북과 같은 위치에 둡니다.
`prepared/한국로텍_분석참고자료.json`도 필요합니다. 서버 이용자는 운영자가 배치한 자료를 사용합니다.

## 실행 방법
1. 검증한 실행 환경은 **Python 3.14**입니다. 현재 배포본은 같은 환경을 재현하도록 버전을 확인합니다.
2. 첫 코드 셀부터 위에서부터 순서대로 실행하거나 **Run → Run All Cells**를 누릅니다.
3. 첫 코드 셀에서 필요한 라이브러리를 설치합니다. 첫 설치에는 인터넷이 필요합니다.
4. 결과는 `분석결과/한국로텍_분석보고서.html`에서 확인합니다.

커널 재시작 안내가 나오면 재시작한 뒤 처음부터 실행합니다.
세부 계산표는 `분석결과/tables`, 그래프는 `분석결과/figures`에 저장합니다.
다시 실행하면 같은 결과 폴더를 갱신하므로 이전 결과가 필요하면 먼저 복사합니다. 원본 입력자료는 변경하지 않습니다.

## 처음 사용하는 Windows PC
자료를 배치한 폴더에서 PowerShell을 열어 다음 명령을 한 줄씩 실행합니다.

```powershell
py -3.14 -m venv .venv
.\.venv\Scripts\python.exe -c "import json;n=json.load(open('한국로텍_데이터분석.ipynb',encoding='utf-8'));c=next(c for c in n['cells'] if 'rotec-setup' in c.get('metadata',{}).get('tags',[]));exec(''.join(c['source']))"
.\.venv\Scripts\python.exe -m notebook "한국로텍_데이터분석.ipynb"
```

노트북이 열리면 **Python 3 (ipykernel)**을 선택합니다. 실행 중에는 PowerShell 창을 열어 둡니다.
자세한 설치 방법은 저장소의 [Windows 가이드](https://github.com/realironn/Hkrotec-Data-Analysis/blob/main/docs/windows-setup.md)를 참고해 주세요.


In [ ]:
# 필요한 라이브러리를 현재 Python 환경에 설치합니다.
import sys, json, subprocess
from pathlib import Path

if sys.version_info[:2] != (3, 14):
    raise RuntimeError('Python 3.14 환경을 선택해 주세요. 처음 사용하는 PC의 준비 방법은 노트북 맨 위에 있습니다.')

_nb_name = '한국로텍_데이터분석.ipynb'
_nb_candidates = [p/q/_nb_name for p in [Path.cwd(), *Path.cwd().parents]
                  for q in [Path('.'), Path('04_노트북_실행용'), Path('한국로텍_최종/04_노트북_실행용')]]
_nb_file = next((p for p in _nb_candidates if p.is_file()), None)
if _nb_file is None:
    raise FileNotFoundError('작업 위치를 노트북이 있는 폴더로 맞춘 뒤 다시 실행해 주세요.')
_runtime = json.loads(_nb_file.read_text(encoding='utf-8'))['metadata']['rotec_runtime']
_install_packages = _runtime['requirements_lock']
if not _install_packages or not all(isinstance(p,str) and '==' in p for p in _install_packages):
    raise ValueError('노트북의 설치 목록을 확인해 주세요.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       '--disable-pip-version-check', '--quiet', *_install_packages])
print('필요 라이브러리 준비 완료. 다음 셀부터 순서대로 실행해 주세요.')


## 0. 입력자료 준비
원본 파일과 저장된 이미지 분류표를 대조합니다. 이미지 파일과 숫자 셀의 일치 여부를 확인합니다.


In [ ]:
"""입력파일과 이미지 분류표를 대조하고 분석에 필요한 파일을 준비합니다."""
from pathlib import Path
import base64,csv,hashlib,io,json,zipfile

def _file_hash(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream,'sha256').hexdigest()

def _prepare_original_inputs(root,out,runtime):
    root=Path(root).resolve();out=Path(out).resolve()
    if not out.is_relative_to(root):raise ValueError('결과는 입력 폴더 안에 저장해 주세요.')
    out.mkdir(parents=True,exist_ok=True)
    reference_path=root/runtime['private_reference_file']
    if not reference_path.is_file():
        raise FileNotFoundError('분석참고자료 파일이 없습니다. 별도로 받은 한국로텍_분석참고자료.json을 prepared 폴더 안에 넣어 주세요.')
    reference_bytes=reference_path.read_bytes()
    if hashlib.sha256(reference_bytes).hexdigest()!=runtime['private_reference_sha256']:
        raise ValueError('분석참고자료의 버전이 노트북과 다릅니다. 담당자가 제공한 같은 배포본의 파일인지 확인해 주세요.')
    reference=json.loads(reference_bytes)
    bundle=reference['embedded_review']
    payload=base64.b64decode(bundle['zip_base64'],validate=True)
    if hashlib.sha256(payload).hexdigest()!=bundle['zip_sha256']:
        raise ValueError('노트북 내부 검토자료가 손상되었습니다.')
    written=[]
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        for member in archive.infolist():
            if member.is_dir():continue
            relative=Path(member.filename)
            if relative.is_absolute() or '..' in relative.parts:
                raise ValueError('잘못된 내부 검토자료 경로입니다.')
            base=out
            target=(base/relative).resolve()
            if not target.is_relative_to(base):raise ValueError('실행자료 경로를 확인해 주세요.')
            target.parent.mkdir(parents=True,exist_ok=True)
            target.write_bytes(archive.read(member))
            written.append(target)
    sources={}
    embedded_sources=bundle['embedded_source_paths']
    reference_usage=[]
    failures=[]
    for item in bundle['expected_inputs']:
        rel=item['file'];actual=(root/rel).resolve()
        if not actual.is_relative_to(root):raise ValueError('입력 경로가 분석 폴더 밖입니다.')
        mode='받은 입력자료'
        if not actual.is_file() and rel in embedded_sources:
            actual=(out/embedded_sources[rel]).resolve()
            if not actual.is_relative_to(out):raise ValueError('내장 기준자료 경로 오류')
            mode='노트북에 포함한 기존 과제·테마 기준'
        if not actual.is_file():failures.append(rel+' : 파일 없음');continue
        if _file_hash(actual)!=item['sha256']:
            failures.append(rel+' : 검토 기준과 내용 다름');continue
        sources[rel]=actual
        if rel in embedded_sources:
            reference_usage.append({'기준자료':rel,'사용위치':actual.relative_to(root).as_posix(),'구분':mode,'SHA256':item['sha256']})
    if failures:
        raise ValueError('기존 분석과 대조할 입력자료를 확인해 주세요. 원본·csv·prepared는 변경하지 않습니다.\n'+'\n'.join(failures[:12]))
    expected_assets=bundle['expected_image_assets']
    image_dir=out/'이미지검토'
    assets_dir=image_dir/'01_추출이미지'
    # 이전 실행의 파일이 모두 같은 원본 이미지라면 추출 단계를 재사용합니다.
    cached=all((assets_dir/item['filename']).is_file() and _file_hash(assets_dir/item['filename'])==item['sha256'] for item in expected_assets)
    if not cached:
        _extract_original_images(root,image_dir,bundle['review_inventory'],sources)
    actual_assets={p.name:_file_hash(p) for p in assets_dir.iterdir() if p.is_file()}
    if any(actual_assets.get(item['filename'])!=item['sha256'] for item in expected_assets):
        raise ValueError('원본에서 다시 추출한 이미지가 기존 검토 기록과 다릅니다. 이미지 재검토가 필요합니다.')
    from PIL import Image
    rendered=image_dir/'02_열람용이미지';rendered.mkdir(exist_ok=True)
    fallback=set(bundle['embedded_image_renders'])
    for item in expected_assets:
        original=assets_dir/item['filename'];target=rendered/(original.stem+'.png')
        if original.name in fallback:
            if not target.is_file():raise ValueError('검증된 열람 이미지가 없습니다: '+original.name)
            continue
        with Image.open(original) as picture:
            picture.convert('RGBA').save(target,format='PNG')
    source_note={
        '검사방식':'원본 파일 해시, 추출 이미지, 숫자 셀 대조',
        '자동_OCR_실행':False,
        '원본이미지_해시대조수':len(expected_assets),
        '검토기준자료_원본수':len(bundle['review_inventory']),
        '기준문서_사용내역':reference_usage,
    }
    (image_dir/'현재실행_근거설명.json').write_text(json.dumps(source_note,ensure_ascii=False,indent=2),encoding='utf-8')
    pd.DataFrame(reference_usage).to_csv(out/'tables/실행기준_사용내역.csv',index=False,encoding='utf-8-sig')
    return {'sources':sources,'image_audit':image_dir,'parameters':reference['analysis_parameters'],
            'review_inventory':bundle['review_inventory'],'source_note':source_note}



def _extract_original_images(root,output,review_inventory,sources):
    """원본 95개에서 삽입 이미지 추출 및 위치 추적. Office 파일을 저장하지 않는다."""
    from pathlib import Path
    from collections import Counter, defaultdict
    import csv, hashlib, io, json, posixpath, struct, zipfile, zlib, re, os
    import xml.etree.ElementTree as ET
    from PIL import Image, ImageOps, ImageDraw, ImageFont
    from xlrd.compdoc import CompDoc

    ROOT=Path(root)
    OUT=Path(output)
    ASSETS = OUT / '01_추출이미지'
    WORK=OUT/'05_재실행도구'
    for p in [OUT,ASSETS,WORK]:p.mkdir(parents=True,exist_ok=True)
    NS = {'r':'http://schemas.openxmlformats.org/officeDocument/2006/relationships',
          'a':'http://schemas.openxmlformats.org/drawingml/2006/main',
          'xdr':'http://schemas.openxmlformats.org/drawingml/2006/spreadsheetDrawing',
          's':'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
    inventory=review_inventory
    assets, occurrences, file_audit, external, embeddings = {}, [], [], [], []

    def rel_source(rel_path):
        if rel_path == '_rels/.rels': return ''
        parent, leaf = posixpath.split(rel_path)
        return posixpath.join(posixpath.dirname(parent), leaf[:-5])

    def sha(data): return hashlib.sha256(data).hexdigest()

    def image_type(data, fallback=''):
        if data.startswith(b'\x89PNG\r\n\x1a\n'): return '.png'
        if data.startswith(b'\xff\xd8\xff'): return '.jpg'
        if data.startswith((b'GIF87a',b'GIF89a')): return '.gif'
        if data.startswith(b'BM'): return '.bmp'
        if data.startswith((b'II*\x00',b'MM\x00*')): return '.tif'
        return fallback or '.bin'

    def register(data, suffix, source, media, locations):
        h = sha(data)
        if h not in assets:
            uid = f'IMG-{len(assets)+1:03d}'
            suffix = image_type(data,suffix.lower())
            dst = ASSETS / (uid+suffix)
            dst.write_bytes(data)
            size, mode, error = None, None, ''
            try:
                with Image.open(io.BytesIO(data)) as im: size,mode=im.size,im.mode
            except Exception as e: error=str(e)
            assets[h] = {'id':uid,'sha256':h,'path':str(dst.relative_to(OUT)),'bytes':len(data),'width':size[0] if size else None,'height':size[1] if size else None,'mode':mode,'read_error':error}
        for loc in locations or ['참조 위치 미확인']:
            occurrences.append({'image_id':assets[h]['id'],'source_file':source,'container_media':media,'location':loc,'sha256':h})

    def biff_pictures(data, source):
        stream = CompDoc(data).get_named_stream('Workbook')
        if stream is None: stream = CompDoc(data).get_named_stream('Book')
        if stream is None: raise ValueError('Workbook/Book stream absent')
        counts, blocks, pos, current, hf = Counter(), [], 0, None, bytearray()
        while pos+4<=len(stream):
            kind,length=struct.unpack_from('<HH',stream,pos); pos+=4
            body=stream[pos:pos+length];pos+=length
            counts[f'0x{kind:04X}']+=1
            if kind==0x00EB:
                if current is None:current=bytearray();blocks.append(current)
                current.extend(body)
            elif kind==0x003C and current is not None:
                current.extend(body)
            else: current=None
            if kind==0x0866:hf.extend(body)
        (WORK/(Path(source).stem+'_biff_records.json')).write_text(json.dumps(dict(counts),indent=2),encoding='utf-8')
        if hf:blocks.append(hf)
        blip_count=0
        declared_blips=0
        for block_no,block in enumerate(blocks):
            for store in re.finditer(b'\x01\xf0',block):
                q=store.start()-2
                if q>=0 and q+8<=len(block):
                    vi,kt,ln=struct.unpack_from('<HHI',block,q)
                    if (vi&15)==15 and ln<=len(block)-q-8:declared_blips+=vi>>4
            for hit in re.finditer(b'[\x1a-\x1f\x29]\xf0',block):
                q=hit.start()-2
                if q<0 or q+8>len(block):continue
                verinst,kind,length=struct.unpack_from('<HHI',block,q)
                if length>len(block)-q-8:continue
                body=bytes(block[q+8:q+8+length])
                if kind in (0xF01D,0xF01E,0xF01F,0xF029):
                    for sig,suffix in [(b'\x89PNG\r\n\x1a\n','.png'),(b'\xff\xd8\xff','.jpg')]:
                        off=body.find(sig,0,80)
                        if off>=0:
                            register(body[off:],suffix,source,f'BIFF/OfficeArt/block{block_no}/0x{kind:04X}',['BIFF 도형/머리글; 시트 위치 미확인'])
                            blip_count+=1
                            break
                    else: embeddings.append({'source_file':source,'part':f'OfficeArt 0x{kind:04X}','bytes':length,'issue':'Raster BLIP without expected signature'})
                elif kind in (0xF01A,0xF01B,0xF01C):
                    decoded=False
                    for uidbytes in [16,32]:
                        if len(body)<uidbytes+34:continue
                        cbsize=struct.unpack_from('<I',body,uidbytes)[0]
                        cbsave=struct.unpack_from('<I',body,uidbytes+28)[0]
                        compression,filterbyte=body[uidbytes+32:uidbytes+34]
                        payload=body[uidbytes+34:uidbytes+34+cbsave]
                        if filterbyte!=0xFE or compression not in (0,0xFE) or len(payload)!=cbsave:continue
                        try:picture=zlib.decompress(payload) if compression==0 else payload
                        except zlib.error:continue
                        if len(picture)!=cbsize:continue
                        register(picture,{0xF01A:'.emf',0xF01B:'.wmf',0xF01C:'.pict'}[kind],source,f'BIFF/OfficeArt/block{block_no}/0x{kind:04X}',['BIFF 도형/머리글; 시트 위치 미확인'])
                        blip_count+=1;decoded=True;break
                    if not decoded:embeddings.append({'source_file':source,'part':f'OfficeArt 0x{kind:04X}','bytes':length,'issue':'Metafile BLIP; further decode required'})
        return {'biff_records':sum(counts.values()),'drawing_groups':counts['0x00EB'],'drawing_records':counts['0x00EC'],'legacy_image_records':counts['0x007F'],'header_image_records':counts['0x0866'],'declared_blips':declared_blips,'extracted_blips':blip_count}

    for inv in inventory:
        rel=inv['source_file'];p=sources[rel];data=p.read_bytes()
        assert sha(data)==inv['sha256'],f'original changed: {rel}'
        before=len(occurrences)
        row={'source_file':rel,'extension':p.suffix.lower(),'media_parts':0,'sha256_matches':True,'errors':''}
        if zipfile.is_zipfile(io.BytesIO(data)):
            with zipfile.ZipFile(io.BytesIO(data)) as z:
                names=set(z.namelist());rels={};reverse=defaultdict(set);roots={};anchors=defaultdict(list)
                for name in sorted(names):
                    if not name.endswith('.rels'):continue
                    parent=rel_source(name);mapping={}
                    for node in ET.fromstring(z.read(name)):
                        if node.attrib.get('TargetMode')=='External':
                            if '/image' in node.attrib.get('Type',''):external.append({'source_file':rel,'part':parent,**node.attrib})
                            continue
                        target=posixpath.normpath(posixpath.join(posixpath.dirname(parent),node.attrib['Target'].lstrip('/'))) if not node.attrib['Target'].startswith('/') else node.attrib['Target'].lstrip('/')
                        mapping[node.attrib['Id']]=target;reverse[target].add(parent)
                    rels[parent]=mapping
                if 'xl/workbook.xml' in names:
                    for s in ET.fromstring(z.read('xl/workbook.xml')).findall('s:sheets/s:sheet',NS):
                        target=rels.get('xl/workbook.xml',{}).get(s.attrib['{'+NS['r']+'}id'])
                        if target:roots[target]='시트: '+s.attrib['name']
                for name in names:
                    if name.startswith('ppt/slides/slide') and name.endswith('.xml') and '/_rels/' not in name:
                        roots[name]='슬라이드 '+Path(name).stem.replace('slide','')
                    if name.startswith('xl/drawings/drawing') and name.endswith('.xml'):
                        for anchor in ET.fromstring(z.read(name)):
                            origin=anchor.find('xdr:from',NS)
                            loc=''
                            if origin is not None:
                                col=int(origin.find('xdr:col',NS).text);rr=int(origin.find('xdr:row',NS).text)+1
                                cc='';col+=1
                                while col:col,rem=divmod(col-1,26);cc=chr(65+rem)+cc
                                loc=cc+str(rr)
                            for b in anchor.findall('.//a:blip',NS):
                                target=rels.get(name,{}).get(b.attrib.get('{'+NS['r']+'}embed',''))
                                if target:anchors[target].append((name,loc))
                def trace_roots(target):
                    found=set();seen=set();queue=[target]
                    while queue:
                        node=queue.pop()
                        if node in seen:continue
                        seen.add(node)
                        if node in roots:found.add(roots[node])
                        queue.extend(reverse.get(node,()))
                    return found
                def locations(target):
                    if anchors.get(target):
                        return sorted({loc+' / '+a for drawing,a in anchors[target] for loc in trace_roots(drawing)})
                    found=trace_roots(target)
                    return sorted(found) or (['문서 축소판'] if target.startswith('docProps/') else ['참조 위치 미확인'])
                media=[n for n in sorted(names) if '/media/' in n or n.startswith('docProps/thumbnail')]
                row['media_parts']=len(media)
                for name in media:register(z.read(name),Path(name).suffix,rel,name,locations(name))
                for name in sorted(names):
                    if '/embeddings/' in name:
                        nested=z.read(name);nested_media=[]
                        if zipfile.is_zipfile(io.BytesIO(nested)):
                            with zipfile.ZipFile(io.BytesIO(nested)) as nz:
                                for nn in nz.namelist():
                                    if '/media/' in nn:
                                        nested_media.append(nn);register(nz.read(nn),Path(nn).suffix,rel,name+'/'+nn,['내장 문서 / '+name])
                        embeddings.append({'source_file':rel,'part':name,'bytes':len(nested),'issue':'Embedded Office document; checked','image_parts':nested_media})
        elif p.suffix.lower()=='.xls':
            row.update(biff_pictures(data,rel))
        else: row['errors']='Unrecognized file container'
        row['image_occurrences']=len(occurrences)-before
        file_audit.append(row)

    result={'files':file_audit,'assets':list(assets.values()),'occurrences':occurrences,'external_images':external,'embedded_objects':embeddings}
    (WORK/'extraction.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
    print(json.dumps({'files_checked':len(file_audit),'files_with_images':sum(r['image_occurrences']>0 for r in file_audit),'unique_images':len(assets),'image_occurrences':len(occurrences),'extensions':dict(Counter(Path(a['path']).suffix for a in assets.values())),'external_images':len(external),'embedded_objects':len(embeddings),'xls_drawing_groups':sum(r.get('drawing_groups',0) for r in file_audit),'xls_legacy_images':sum(r.get('legacy_image_records',0) for r in file_audit),'xls_header_images':sum(r.get('header_image_records',0) for r in file_audit)},ensure_ascii=False))
    print('XLS BLIP checks',Counter((r.get('declared_blips'),r.get('extracted_blips')) for r in file_audit if r['extension']=='.xls'))

    return result


In [ ]:
from pathlib import Path
from datetime import datetime
import csv, re, json, hashlib, sys, platform, base64, html, warnings, zipfile, os, shutil
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display, Markdown, Image
from openpyxl import load_workbook
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

_required_inputs=['raw','csv/manifest.csv','prepared/five_fold/daily_targets.csv','prepared/five_fold/fold_assignments.csv']
_root_candidates=[q for p in [_nb_file.parent,Path.cwd(),*Path.cwd().parents]
                  for q in [p,p/'04_노트북_실행용',p/'한국로텍_최종/04_노트북_실행용']]
ROOT=next((p.resolve() for p in _root_candidates if all((p/r).exists() for r in _required_inputs)),None)
if ROOT is None:
    raise FileNotFoundError('raw·csv·prepared 폴더와 노트북을 같은 위치에 두세요.')
_default_output='분석결과_재실행' if (ROOT/'03_검증도구').is_dir() else '분석결과'
OUT=Path(os.environ.get('ROTEC_RESULTS_DIR',ROOT/_default_output)).resolve()
if not OUT.is_relative_to(ROOT):raise ValueError('출력은 현재 입력 폴더 안에 지정하세요.')
TABLES,FIGURES=OUT/'tables',OUT/'figures'
for p in [OUT,TABLES,FIGURES]:p.mkdir(parents=True,exist_ok=True)
_PREPARED_INPUTS=_prepare_original_inputs(ROOT,OUT,_runtime)
IMAGE_AUDIT=_PREPARED_INPUTS['image_audit']
def _input_file(relative):
    return _PREPARED_INPUTS['sources'].get(str(relative),ROOT/relative)

ITEMS = list(_PREPARED_INPUTS['parameters']['items'])
LAGS = [0,5,10]  # 검사일 t와 공정일 t-lag의 달력 날짜를 정확하게 연결
VOLTAGE_FLOOR = 0.1  # 진단용 가정. 원자료는 유지하고 <=0.1V 제외 민감도를 함께 제시
STATION_TO_LINE = {} # 확인된 설비·액조 대응이 있는 경우에만 사용합니다.
SEED = 20261012
np.random.seed(SEED)
font_path = Path('C:/Windows/Fonts/malgun.ttf')
if font_path.exists():
    font_manager.fontManager.addfont(str(font_path))
    plt.rcParams['font.family']='Malgun Gothic'
else:
    available_fonts={font.name for font in font_manager.fontManager.ttflist}
    selected_font=next((name for name in ['Noto Sans CJK KR','Noto Sans CJK JP','NanumGothic'] if name in available_fonts),None)
    if selected_font:plt.rcParams['font.family']=selected_font
plt.rcParams.update({'axes.unicode_minus':False,'figure.dpi':130,'axes.spines.top':False,'axes.spines.right':False})
pd.set_option('display.max_columns',18)

def save(df,name):
    df.to_csv(TABLES/(name+'.csv'),index=False,encoding='utf-8-sig',date_format='%Y-%m-%d')
    return df
def rows_for(path):
    with Path(path).open(encoding='utf-8-sig',newline='') as f: return list(csv.reader(f))
def num(v):
    try: return float(str(v).replace(',','').replace('₩','').strip())
    except (ValueError,TypeError): return np.nan
def isdate(v): return bool(re.match(r'^20\d{2}-\d{2}-\d{2}',str(v)))
def parts(v): return [x.strip() for x in str(v).replace('_x000D_','').splitlines() if x.strip()]
def slot(v,k):
    p=parts(v); return num(p[k]) if len(p)>k else np.nan
def sec(v):
    s=str(v).strip(); m=re.fullmatch(r'(-?)(\d+):(\d{2}):(\d{2})',s)
    return (-1 if m[1] else 1)*(int(m[2])*3600+int(m[3])*60+int(m[4])) if m else np.nan
def digest(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
manifest=pd.read_csv(ROOT/'csv/manifest.csv',keep_default_na=False)
display(Markdown('**입력 확인:** '+str(ROOT)))

## 1. 분석대상과 계산 기준
테마 문서의 FRT 1품번과 RR 2품번을 각각 분석합니다.
핀홀·피트는 원본 검사표에서 하나의 불량 항목으로 집계합니다.
전류밀도는 설정전류를 표면적으로 나누어 계산하고, 실제 평균전류 기준 값도 함께 비교합니다.


In [ ]:
requirements=pd.DataFrame([
 ['대표품목','; '.join(ITEMS),'기업 확정','테마-1 A9:A10 (RR는 2개 실제 품번)'],
 ['대상 불량','도금-핀홀/피트','기업 확정','테마-1 A17; 검사자료 코드21'],
 ['두께','다른 6품번 두께 기록 21개·시트 SPEC 존재. 대표 3품번 실측 0개, 단위 미표기','표본·정의 제한','2026년 액관리 이력대장 마이크로크랙 3시트 L열; 검사 코드26'],
 ['전류밀도','설정전류(A)/기하학적 표면(dm²); 실제평균도 보조 계산','기업 수식','테마-1 A13 및 삽입 이미지'],
 ['시차','검사일-공정일 = 0,5,10 달력일','실행과제','한국로텍 실행과제.txt'],
 ['불량 지표','불량건수/검사수량×100 (PPHV)','원자료 계산 재현','중복불량 여부 미확정. 고유 불량품률로 단정하지 않음'],
 ['공정-액조 매핑','스테이션 14/15/16과 액관리 1/3/4라인의 대응 없음','미확정','확정 후에만 개별 도금액 노출량 연결 가능'],
 ['합격·경보 기준','대표품목 판정기준·관리 조치 임계값 미확정. 기타품목의 시트 SPEC 존재','미확정','상관관계로 생산조건 또는 합격기준을 새로 확정하지 않음'],
 ['기존 분석','prepared/five_fold의 표본 및 집계 재현 가능; 기존 모델·계수·결과 보고서 없음','부분 재현','206일 표본 재현과 기존 모델 결과 재현을 구분'],
],columns=['항목','정의','상태','근거'])
save(requirements,'requirements');display(requirements)
theme_path=_input_file('raw/추가전달자료/2026-10-07/PBL과제 테마 재선정 및 적용방법 리스트_26.10.07.xlsx')
theme_wb=load_workbook(theme_path,read_only=True,data_only=False)
theme_cells=pd.DataFrame([{'source_file':theme_path.relative_to(ROOT).as_posix(),'sheet':ws.title,'cell':c.coordinate,'text':str(c.value)} for ws in theme_wb for row in ws for c in row if c.value is not None])
theme_wb.close();save(theme_cells,'supplement_theme_source_cells')
theme_reference=pd.DataFrame([
 ['테마-1','A7:A10','FRT/RR 대표품목','FRT 1품번, RR 2품번을 각각 추출','representative_chrome_events.csv'],
 ['테마-1','A12:A13 및 삽입 이미지','전류밀도·전압·시간','설정 전류밀도가 주지표, 실제평균은 보조지표. 이미지 산술 오류는 공식에 따라 계산','representative_process_statistics.csv'],
 ['테마-1','A14:A16','크롬산·철분·3가크롬·황산 및 A/B/TCC','성분과 첨가제 실측 농도 분석. 적용기간별 기준·공정-액조 대응은 미확정','solution_limits_summary.csv; additive_lab_coverage.csv'],
 ['테마-1','A17:A19','핀홀·피트와 공정·성분의 관계','검사 원본은 핀홀/피트 통합코드. 품번·LOT 부재로 날짜 탐색만 가능','lag_correlations.csv'],
 ['테마-1','E46:E50','불순물 상승 → 전압 상승, 밀도 감소·시간 증가 가설','설정 전류밀도 통제 및 실제평균 통제 민감도 비교. 인과·운영조건 확정 불가','field_feedback_revalidation.csv'],
 ['테마-1','Q3 인근 삽입 [예시] 표','품목별 일자 공정조건+도금액+불량 통합표의 예시','7/1~7/7 숫자는 [예시]로 표시된 설명 자료. 실제 관측/검사/두께 데이터나 확정 판정기준으로 사용하지 않음','supplement_theme_review.csv'],
 ['테마-2','A8:A11','에칭·도금 크롬산과 첨가제 투입량/전력량','월별 순투입/자동투입 대비 kWh와 Ah를 구분. 첨가제는 Nr.6 단일 투입계열','monthly_chemical_energy_ratios.csv; monthly_theme2_summary.csv'],
 ['테마-2','A8, A12','첨가제 2종 구분 및 적정 투입기준','로그에서는 Nr.6 한 계열만 구분되며 A/B/TCC와의 제품 대응은 미확정','monthly_theme2_summary.csv'],
],columns=['시트','원본 위치','요구사항','반영 및 한계','근거 파일'])
save(theme_reference,'supplement_theme_review')

## 2. 입력자료와 표본 수
파일 수, 기간, 열 이름과 분석에 사용한 기록 수를 정리합니다. 서식의 빈 행과 실제 기록을 구분합니다.


In [ ]:
originals=sorted(p for p in (ROOT/'raw').rglob('*') if p.is_file())
inventory=pd.DataFrame([{'source_file':p.relative_to(ROOT).as_posix(),'bytes':p.stat().st_size,'sha256':digest(p)} for p in originals])
initial_hashes=dict(zip(inventory.source_file,inventory.sha256))
save(inventory,'source_files_sha256')
duplicates=[]
for copy_dir_name in ['도금액분석기록','생산작업이력','약품투입기록','월별 불량현황','일별 불량현황','전력사용기록']:
    p=ROOT/copy_dir_name
    if p.is_dir():
        for f in p.rglob('*'):
            if f.is_file():
                h=digest(f); same=inventory[inventory.sha256.eq(h)]
                duplicates.append({'copy':f.relative_to(ROOT).as_posix(),'canonical':';'.join(same.source_file),'hash_equal':len(same)>0})
duplicates=pd.DataFrame(duplicates,columns=['copy','canonical','hash_equal'])
save(duplicates,'duplicate_copies')
input_fingerprints={p.relative_to(ROOT).as_posix():digest(p) for p in sorted((ROOT/'csv').rglob('*.csv'))+sorted((ROOT/'prepared').rglob('*.csv'))+([ROOT/'한국로텍 실행과제.txt'] if (ROOT/'한국로텍 실행과제.txt').is_file() else [])}
schema=[]
for _,r in manifest.iterrows():
    a=rows_for(ROOT/r.csv_relative_path) if r.source_group!='production_history' else None
    if a is None:
        with (ROOT/r.csv_relative_path).open(encoding='utf-8-sig',newline='') as f: a=[next(csv.reader(f))]
    labels=a[0] if r.source_group=='production_history' else [v for row in a[:40] for v in row[:8] if v and not isdate(v)]
    schema.append({'group':r.source_group,'source_file':r.source_relative_path,'table':r.table_name,'csv':r.csv_relative_path,'converted_rows':r.row_count,'converted_columns':r.column_count,'key_labels':' ; '.join(dict.fromkeys(labels))[:1600]})
schema=pd.DataFrame(schema);save(schema,'source_tables_and_columns')
display(manifest.groupby('source_group').agg(files=('source_relative_path','nunique'),tables=('csv_relative_path','size'),converted_rows=('row_count','sum')))
print('원본 파일',len(inventory),'중복 사본',len(duplicates))

## 3. 검사 집계
해당 월의 날짜 열만 집계합니다. 검사수량이 빈 6일은 결측으로 유지하며 0%로 계산하지 않습니다.


In [ ]:
daily_rows, defect_rows, daily_audit, raw_checks=[],[],[],[]
for _,r in manifest[(manifest.source_group=='daily_defects')&manifest.table_name.eq('구간 1')].iterrows():
    a=rows_for(ROOT/r.csv_relative_path)
    yy,mm=map(int,re.search(r'(20\d{2})년_(\d{2})월',r.csv_relative_path).groups())
    h=next(i for i,row in enumerate(a) if row[0]=='Problem Description')
    total=next(i for i,row in enumerate(a[h+1:],h+1) if row[0]=='Total')
    reviewed=next(i for i,row in enumerate(a[h+1:],h+1) if row[0]=='Total Reviewed')
    pphvrow=next(i for i,row in enumerate(a[h+1:],h+1) if row[0]=='PPHV')
    excluded=[]; current=[]
    for j,v in enumerate(a[h]):
        if not isdate(v):continue
        date=pd.Timestamp(v).normalize()
        if date.year!=yy or date.month!=mm: excluded.append(str(date.date()));continue
        ds=[]
        for i in range(h+1,total):
            val=0. if a[i][j].strip()=='' else num(a[i][j])
            defect_rows.append({'date':date,'defect_type':a[i][0],'count':val,'source_file':r.source_relative_path,'source_row':i+1,'source_col':j+1})
            ds.append(val)
        n=num(a[reviewed][j]); d=num(a[total][j]); p=num(a[pphvrow][j])
        daily_rows.append({'date':date,'total_defects':d,'total_reviewed':n,'pphv_source':p,'pphv':d/n*100 if n>0 else np.nan,'defect_type_sum':sum(ds),'source_file':r.source_relative_path,'source_col':j+1})
        current.append(j)
    daily_audit.append({'month':f'{yy}-{mm:02d}','title':a[1][6],'date_columns':len(current),'excluded_other_month_dates':';'.join(excluded),'sheet_total':num(a[total][5]),'date_column_total':sum(num(a[total][j]) for j in current),'sheet_reviewed':num(a[reviewed][5]),'date_column_reviewed':float(np.nansum([num(a[reviewed][j]) for j in current])),'missing_reviewed_dates':sum(pd.isna(num(a[reviewed][j])) for j in current),'formula_error_cells':sum(v.startswith(('#REF!','#DIV/0!','#VALUE!')) for row in a for v in row)})
    # 선택한 날짜의 숫자를 원본 셀과 대조합니다.
    wb=load_workbook(ROOT/r.source_relative_path,read_only=True,data_only=True)
    ws=wb['구간 1']; vals=list(ws.values); mismatch=0; tested=0
    for i in [h,*range(h+1,total),total,reviewed,pphvrow]:
        for j in current:
            x=vals[i][j] if j<len(vals[i]) else None; y=a[i][j];tested+=1
            if isinstance(x,datetime): ok=pd.Timestamp(x)==pd.Timestamp(y)
            elif x is None: ok=y==''
            elif isinstance(x,(int,float)): ok=np.isclose(float(x),num(y),equal_nan=True)
            else:ok=str(x)==y
            mismatch+=not ok
    raw_checks.append({'source_file':r.source_relative_path,'cells_checked':tested,'mismatches':mismatch});wb.close()
daily=pd.DataFrame(daily_rows).sort_values('date').reset_index(drop=True)
defects=pd.DataFrame(defect_rows)
pivot=defects.pivot(index='date',columns='defect_type',values='count')
daily=daily.merge(pivot[['도금-핀홀/피트','도금두께미달']],on='date',validate='one_to_one').rename(columns={'도금-핀홀/피트':'pinhole_pit_count','도금두께미달':'thickness_under_count'})
for target in ['pinhole_pit','thickness_under']:
    daily[target+'_pphv']=daily[target+'_count']/daily.total_reviewed.where(daily.total_reviewed>0)*100
daily['month']=daily.date.dt.strftime('%Y-%m')
valid=daily[daily.total_reviewed.gt(0)].copy()
prepared=pd.read_csv(ROOT/'prepared/five_fold/daily_targets.csv',parse_dates=['date'])
repro=daily.merge(prepared,on='date',suffixes=('_rebuilt','_prepared'),validate='one_to_one')
repro['count_difference']=repro.total_defects_rebuilt-repro.total_defects_prepared
repro['reviewed_difference']=repro.total_reviewed_rebuilt-repro.total_reviewed_prepared
repro['valid_denominator']=repro.total_reviewed_rebuilt.gt(0)
repro['prepared_zero_for_source_blank']=repro.total_reviewed_rebuilt.isna()&repro.total_reviewed_prepared.eq(0)
save(daily,'daily_inspection_rebuilt');save(defects,'defect_counts_rebuilt');save(pd.DataFrame(daily_audit),'inspection_sheet_audit');save(pd.DataFrame(raw_checks),'original_cell_checks');save(repro,'reproduction_comparison')
monthly=daily.groupby('month').agg(calendar_days=('date','size'),valid_days=('total_reviewed',lambda s:int((s>0).sum())),defects=('total_defects','sum'),reviewed=('total_reviewed','sum'),pinhole_pit=('pinhole_pit_count','sum'),thickness_under=('thickness_under_count','sum')).reset_index()
monthly['total_pphv']=100*monthly.defects/monthly.reviewed;monthly['pinhole_pit_pphv']=100*monthly.pinhole_pit/monthly.reviewed
save(monthly,'monthly_inspection');display(monthly)
print('수령 날짜',len(daily),'분모>0',len(valid),'검사수량 결측/0',len(daily)-len(valid),'유효일 기존 집계 차이 최대',repro.loc[repro.valid_denominator,['count_difference','reviewed_difference']].abs().max().to_dict())

## 4. 생산·도금 이력
크롬 공정과 대표품목 기록을 추출합니다. 공정 통과행에는 재작업이 포함될 수 있어 제품 수와 구분합니다.
FB는 설비 운행 식별자이며 회사 LOT와 동일하다고 가정하지 않습니다.


In [ ]:
chrome_frames,production_audit,route_frames=[],[],[]
for _,r in manifest[manifest.source_group=='production_history'].iterrows():
    raw_count=0; stations={}; nitems=set(); exact_dupes=0; entirely_blank=0; station_missing_nonempty=0; valid_time_min=None; valid_time_max=None
    for chunk in pd.read_csv(ROOT/r.csv_relative_path,dtype=str,keep_default_na=False,chunksize=100000):
        cols=list(chunk.columns); raw_count+=len(chunk)
        blank=chunk.eq('').all(axis=1); entirely_blank+=int(blank.sum())
        station_missing_nonempty+=int((chunk.iloc[:,2].eq('') & ~blank).sum())
        times=pd.to_datetime(chunk.iloc[:,1],errors='coerce')
        if times.notna().any():
            valid_time_min=times.min() if valid_time_min is None else min(valid_time_min,times.min())
            valid_time_max=times.max() if valid_time_max is None else max(valid_time_max,times.max())
        for st,n in chunk.iloc[:,2].value_counts().items():stations[st]=stations.get(st,0)+int(n)
        nitems.update(chunk.iloc[:,7].unique())
        chosen=chunk[chunk.iloc[:,7].isin(ITEMS)]
        if len(chosen):
            route_frames.append(pd.DataFrame({'item':chosen.iloc[:,7],'fb_key':chosen.iloc[:,3].map(lambda x:parts(x)[1] if len(parts(x))>1 else ''),'station':chosen.iloc[:,2],'source_file':r.source_relative_path}))
        c=chunk[chunk.iloc[:,2].str.contains('크롬',regex=False)].copy()
        if c.empty:continue
        z=pd.DataFrame({'start_time':pd.to_datetime(c.iloc[:,0],errors='coerce'),'end_time':pd.to_datetime(c.iloc[:,1],errors='coerce'),'station':c.iloc[:,2],'fb_raw':c.iloc[:,3],'item':c.iloc[:,7].str.strip(),'item_name':c.iloc[:,8],'parts_count':pd.to_numeric(c.iloc[:,9],errors='coerce'),'galvanic_area_dm2':pd.to_numeric(c.iloc[:,10],errors='coerce'),'geometric_area_dm2':pd.to_numeric(c.iloc[:,11],errors='coerce')})
        z['fb_key']=c.iloc[:,3].map(lambda x:parts(x)[1] if len(parts(x))>1 else '')
        for label,idx in [('current_a',13),('voltage_v',17),('temperature_c',19)]:
            for k,name in enumerate(['set','min','max','avg']):z[label+'_'+name]=c.iloc[:,idx].map(lambda v:slot(v,k))
        z['duration_set_s']=c.iloc[:,5].map(lambda v:sec(parts(v)[0]) if len(parts(v)) else np.nan)
        z['duration_actual_s']=c.iloc[:,5].map(lambda v:sec(parts(v)[1]) if len(parts(v))>1 else np.nan)
        z['duration_timestamp_s']=(z.end_time-z.start_time).dt.total_seconds()
        z['charge_actual_ah']=c.iloc[:,14].map(lambda v:slot(v,1))
        z['date']=z.end_time.dt.normalize()
        z['density_set_a_dm2']=z.current_a_set/z.geometric_area_dm2.where(z.geometric_area_dm2>0)
        z['density_actual_a_dm2']=z.current_a_avg/z.geometric_area_dm2.where(z.geometric_area_dm2>0)
        z['voltage_low_flag']=z.voltage_v_avg.le(VOLTAGE_FLOOR)&z.current_a_avg.gt(0)
        z['voltage_clean_v']=z.voltage_v_avg.mask(z.voltage_low_flag)
        z['source_file']=r.source_relative_path;z['source_row']=c.index+2
        chrome_frames.append(z)
    production_audit.append({'source_file':r.source_relative_path,'data_rows':raw_count,'part_numbers':len(nitems),'station_counts':json.dumps(stations,ensure_ascii=False),'entirely_blank_rows':entirely_blank,'station_missing_nonempty_rows':station_missing_nonempty,'station_present_rows':raw_count-stations.get('',0),'first_valid_end_time':valid_time_min,'last_valid_end_time':valid_time_max})
    print(Path(r.source_relative_path).name,raw_count,'행 읽음')
chrome=pd.concat(chrome_frames,ignore_index=True)
dupekeys=['start_time','end_time','station','fb_raw','item']
duplicate_mask=chrome.duplicated(dupekeys,keep='first')
save(chrome[duplicate_mask],'production_exact_duplicates')
chrome=chrome[~duplicate_mask].copy()
representative=chrome[chrome.item.isin(ITEMS)].copy()
chrome['active_event']=chrome.current_a_avg.gt(0)&chrome.geometric_area_dm2.gt(0)&chrome.duration_actual_s.gt(0)
active_chrome=chrome[chrome.active_event].copy()
active_chrome['temperature_c_avg']=active_chrome.temperature_c_avg.where(active_chrome.temperature_c_avg>0)
active_representative=active_chrome[active_chrome.item.isin(ITEMS)].copy()
routes=pd.concat(route_frames,ignore_index=True)
save(pd.DataFrame(production_audit),'production_file_rows');save(representative,'representative_chrome_events')
route_stats=routes.groupby(['item','fb_key']).station.agg(['size','nunique']).reset_index()
save(route_stats,'representative_fb_route_summary')
process_cols=['density_set_a_dm2','density_actual_a_dm2','current_a_avg','voltage_v_avg','voltage_clean_v','temperature_c_avg','duration_actual_s','duration_set_s','duration_timestamp_s','charge_actual_ah']
item_summary=representative.groupby('item').agg(events=('item','size'),fb_ids=('fb_key','nunique'),first_date=('date','min'),last_date=('date','max'),days=('date','nunique'),logged_parts=('parts_count','sum'),low_voltage_events=('voltage_low_flag','sum')).reset_index()
save(item_summary,'representative_summary')
distributions=active_representative.groupby(['item','station'])[process_cols].agg(['count','mean','std','min','median','max'])
distributions.columns=['__'.join(c) for c in distributions.columns];save(distributions.reset_index(),'representative_process_statistics')
process_daily=active_chrome.groupby('date')[process_cols].mean()
process_daily['chrome_events']=chrome.groupby('date').size()
item_daily=active_representative.groupby(['date','item'])[process_cols].mean().reset_index()
save(process_daily.reset_index(),'process_daily_all_items');save(item_daily,'process_daily_representatives')
display(item_summary)

## 5. 도금액 성분과 관리범위
같은 날짜·라인·성분의 값이 겹치면 관리치 시트를 우선하고 서로 다른 원값은 별도 저장합니다.
크롬산 0으로 채워진 미측정 묶음은 결측 처리합니다. 시트의 규격 범위와 관리 범위는 구분합니다.


In [ ]:
solution_records=[]
components=['크롬산','3가크롬','철분','황산','이물질']
def bounds(v):
    ns=re.findall(r'\d+(?:\.\d+)?',str(v))
    return (float(ns[0]),float(ns[1])) if len(ns)==2 else (np.nan,np.nan)
for _,r in manifest[(manifest.source_group=='plating_solution')&manifest.table_name.isin(['농도','액관리대장','액관리대장(관리치)','액관리대장-6월','액관리대장-7월'])].iterrows():
    a=rows_for(ROOT/r.csv_relative_path); old=(r.table_name=='농도')
    priority={'농도':1,'액관리대장':2,'액관리대장(관리치)':3,'액관리대장-6월':4,'액관리대장-7월':4}[r.table_name]
    dates={}; line=None;bath=None
    for i,row in enumerate(a):
        if sum(isdate(v) for v in row)>5:
            dates={j:pd.Timestamp(v).normalize() for j,v in enumerate(row) if isdate(v)};line=None;continue
        if not dates:continue
        label=row[2] if old else row[0]
        mt=re.search(r'([134])라인',label)
        if mt:line=int(mt[1]);bath='에칭조' if '에칭' in label else '도금조' if '도금' in label else None
        component=row[3] if len(row)>3 else ''
        if line is None or bath is None or component not in components:continue
        slo,shi=(num(row[0]),num(row[1])) if old else bounds(row[1])
        mlo,mhi=(np.nan,np.nan) if old else bounds(row[2])
        for j,date in dates.items():
            value=num(row[j]) if j<len(row) else np.nan
            solution_records.append({'date':date,'line':line,'bath':bath,'component':component,'value_raw':value,'spec_low':slo,'spec_high':shi,'manage_low':mlo,'manage_high':mhi,'priority':priority,'source_file':r.source_relative_path,'source_sheet':r.table_name,'source_row':i+1,'source_col':j+1})
solution_all=pd.DataFrame(solution_records)
solkey=['date','line','bath','component']
conflict_keys=solution_all.groupby(solkey).value_raw.nunique().loc[lambda s:s>1].reset_index()[solkey]
conflicts=solution_all.merge(conflict_keys,on=solkey,how='inner')
save(conflicts,'solution_conflicting_sources')
# 측정값과 시트 우선순위로 선택하며, 하위 시트의 0은 따로 표시합니다.
solution_all['populated']=solution_all.value_raw.notna()
solution=solution_all.sort_values(['populated','priority']).drop_duplicates(solkey,keep='last').copy()
anchor=solution[solution.component.eq('크롬산')][['date','line','bath','value_raw']].rename(columns={'value_raw':'chromic_acid_anchor'})
solution=solution.merge(anchor,on=['date','line','bath'],validate='many_to_one')
solution['unmeasured_group']=solution.chromic_acid_anchor.isna()|solution.chromic_acid_anchor.le(0)
solution['value']=solution.value_raw.mask(solution.unmeasured_group)
solution['outside_spec']=((solution.value<solution.spec_low)|(solution.value>solution.spec_high)).where(solution.value.notna()&solution.spec_low.notna()&solution.spec_high.notna())
solution['outside_management']=((solution.value<solution.manage_low)|(solution.value>solution.manage_high)).where(solution.value.notna()&solution.manage_low.notna()&solution.manage_high.notna())
save(solution,'solution_components_clean')
solution['feature']=solution.bath+'_L'+solution.line.astype(str)+'_'+solution.component
solution_wide=solution.pivot(index='date',columns='feature',values='value')
spec_summary=solution.groupby(['line','bath','component']).agg(measured=('value','count'),spec_evaluable=('outside_spec','count'),spec_exceed=('outside_spec','sum'),management_evaluable=('outside_management','count'),management_exceed=('outside_management','sum')).reset_index()
save(spec_summary,'solution_limits_summary')
lab=[]
for _,r in manifest[(manifest.source_group=='plating_solution')&manifest.table_name.eq('아토텍-분석자료')].iterrows():
    a=rows_for(ROOT/r.csv_relative_path);dates={};last=None;line=None;bath=None
    for j,v in enumerate(a[2]):
        if isdate(v):last=pd.Timestamp(v).normalize()
        if last is not None:dates[j]=last
    for i,row in enumerate(a[4:],4):
        mt=re.search(r'([134])라인',row[0])
        if mt:line=int(mt[1]);bath='에칭조' if '에칭' in row[0] else '도금조'
        if bath!='도금조' or not any(t in row[3] for t in ['ADDITIVE','Additive','TCC']):continue
        for j,date in dates.items():
            if j<len(row) and a[3][j]=='아토텍' and pd.notna(num(row[j])):
                lab.append({'date':date,'line':line,'component':row[3],'value':num(row[j]),'unit':row[2],'range':row[1],'source_file':r.source_relative_path,'source_row':i+1,'source_col':j+1})
lab=pd.DataFrame(lab);save(lab,'additive_lab_measurements')
print('성분 유효 측정',solution.value.notna().sum(),'다중 원천 값 충돌',len(conflict_keys),'첨가제 측정',len(lab))
display(spec_summary[spec_summary.bath.eq('도금조')])

## 6. 약품량과 전력량
약품 일계를 개별 투입내역 합계와 대조합니다. 음수 정정값은 유지하고 순투입량과 자동투입량을 구분합니다.
월별 약품량을 kWh와 Ah로 각각 나누어 비교합니다. kWh와 Ah는 서로 다른 단위입니다.
Nr.6는 기록에서 구분되는 첨가제 한 계열로 분석합니다.


In [ ]:
chem_daily_rows,chem_audits,chem_events=[],[],[]
for _,r in manifest[manifest.source_group=='chemical_inputs'].iterrows():
    a=rows_for(ROOT/r.csv_relative_path)
    nr={'에칭':1,'도금':2,'첨가제':6}[next(k for k in ['에칭','도금','첨가제'] if k in r.csv_relative_path)]
    kind={1:'etch_chromic',2:'chrome_chromic',6:'additive'}[nr]
    day_entries=[];event_entries=[]
    dh=next(row for row in a if row[0]=='날짜 시간')
    daynr=dh.index('Nr.'); dm=next(j for j,v in enumerate(dh) if v.startswith('손 [ml]'))
    da=next(j for j,v in enumerate(dh) if '자동적' in v);dx=next(j for j,v in enumerate(dh) if v.startswith('수동 [ml]'));dc=next(j for j,v in enumerate(dh) if 'Ah/dm' in v)
    eh=next(row for row in a if row[0]=='Nr.' and '날짜 시간' in row)
    et=eh.index('날짜 시간');ev=next(j for j,v in enumerate(eh) if ('총액' in v or '합계 [ml]' in v))
    mh=next(i for i,row in enumerate(a) if row[0]=='Nr.' and '날짜 시간' not in row)
    mt=next(j for j,v in enumerate(a[mh]) if v.startswith('합계 [ml]'))
    for i,row in enumerate(a):
        if isdate(row[0]) and num(row[daynr])==nr:
            entry={'date':pd.Timestamp(row[0]).normalize(),'chemical':kind,'manual_ml':num(row[dm]),'automatic_ml':num(row[da]),'additional_manual_ml':num(row[dx]),'counter_ah_dm2':num(row[dc]),'source_file':r.source_relative_path,'source_row':i+1}
            entry['net_ml']=entry['manual_ml']+entry['automatic_ml']+entry['additional_manual_ml'];day_entries.append(entry)
        if num(row[0])==nr and isdate(row[et]):
            event_entries.append({'date':pd.Timestamp(row[et]).normalize(),'chemical':kind,'event_total_ml':num(row[ev]),'source_file':r.source_relative_path})
    chem_daily_rows.extend(day_entries);chem_events.extend(event_entries)
    monthrow=next(row for row in a[mh+1:mh+6] if num(row[0])==nr)
    chem_audits.append({'source_file':r.source_relative_path,'chemical':kind,'daily_rows':len(day_entries),'events':len(event_entries),'monthly_net_ml':num(monthrow[mt]),'daily_net_sum_ml':sum(x['net_ml'] for x in day_entries),'event_total_sum_ml':sum(x['event_total_ml'] for x in event_entries),'negative_manual_days':sum(x['manual_ml']<0 for x in day_entries)})
chem=pd.DataFrame(chem_daily_rows).sort_values(['date','chemical'])
chem_audit=pd.DataFrame(chem_audits);chem_audit['monthly_minus_daily']=chem_audit.monthly_net_ml-chem_audit.daily_net_sum_ml;chem_audit['daily_minus_events']=chem_audit.daily_net_sum_ml-chem_audit.event_total_sum_ml
save(chem,'chemical_daily');save(chem_audit,'chemical_reconciliation')
chem_wide=chem.pivot(index='date',columns='chemical',values=['net_ml','automatic_ml'])
chem_wide.columns=['dose_'+b+'_'+a for a,b in chem_wide.columns]
power=[]
for _,r in manifest[manifest.source_group=='power_usage'].iterrows():
    a=rows_for(ROOT/r.csv_relative_path);month=pd.Timestamp(a[3][7]).strftime('%Y-%m')
    for i,row in enumerate(a):
        st=num(row[0])
        if st in [13,14,15,16]:power.append({'month':month,'station':int(st),'charge_ah':num(row[6]),'primary_kwh':num(row[11]),'secondary_kwh':num(row[18]),'source_file':r.source_relative_path,'source_row':i+1})
power=pd.DataFrame(power);save(power,'power_monthly_stations')
cm=chem.assign(month=chem.date.dt.strftime('%Y-%m')).groupby(['month','chemical'])[['net_ml','automatic_ml']].sum().reset_index()
pm=power.assign(chemical=np.where(power.station.eq(13),'etch_chromic','chrome_chromic')).groupby(['month','chemical'])[['charge_ah','primary_kwh','secondary_kwh']].sum().reset_index()
energy=cm.merge(pm,on=['month','chemical'],how='left',validate='one_to_one')
energy['net_ml_per_1000ah']=energy.net_ml/energy.charge_ah*1000
energy['net_ml_per_kwh']=energy.net_ml/energy.primary_kwh
energy['automatic_ml_per_1000ah']=energy.automatic_ml/energy.charge_ah*1000
energy['automatic_ml_per_kwh']=energy.automatic_ml/energy.primary_kwh
save(energy,'monthly_chemical_energy_ratios')
theme2_monthly=energy[energy.chemical.ne('additive')].pivot(index='month',columns='chemical',values=['net_ml_per_kwh','automatic_ml_per_kwh'])
theme2_monthly.columns=[a+'__'+b for a,b in theme2_monthly.columns]
theme2_additive=energy[energy.chemical.eq('additive')].set_index('month')[['net_ml','automatic_ml']].add_prefix('additive_')
theme2_monthly=theme2_monthly.join(theme2_additive).reset_index()
save(theme2_monthly,'monthly_theme2_summary')
display(chem_audit[['chemical','daily_rows','events','monthly_minus_daily','daily_minus_events']].groupby('chemical').agg(['sum','max']))

## 7. 기록 연결과 분석 표본
검사자료에는 날짜별 집계만 있고 품번·LOT별 검사수량은 없습니다. 날짜 비교와 LOT 연결을 구분합니다.


In [ ]:
features=process_daily.copy()
for item in ITEMS:
    z=item_daily[item_daily.item.eq(item)].set_index('date')[['density_set_a_dm2','density_actual_a_dm2','voltage_clean_v','temperature_c_avg','duration_actual_s']]
    z.columns=[item+'__'+c for c in z.columns];features=features.join(z,how='outer')
features=features.join(solution_wide,how='outer').join(chem_wide,how='outer')
lab['feature']='lab_L'+lab.line.astype(str)+'_'+lab.component
features=features.join(lab.pivot(index='date',columns='feature',values='value'),how='outer')
for c in chem_wide.columns:
    features[c+'_per_100_events']=features[c]/features.chrome_events.where(features.chrome_events>0)*100
save(features.reset_index(),'daily_features')
join_rows=[{'join':'품번+LOT: 생산→도금→검사','denominator_name':'기업 LOT 모집단(미제공)','denominator':np.nan,'matched':0,'rate':np.nan,'reason':'완전 연결을 확인한 기록 0건. LOT 분모 미정이므로 연결률 산출 불가. 검사 206일을 LOT 분모로 대체하지 않음'},
 {'join':'대표품목 공정4조건 값 존재','denominator_name':'대표품목 크롬 통과행','denominator':len(representative),'matched':int(representative[['density_actual_a_dm2','voltage_clean_v','temperature_c_avg','duration_actual_s']].notna().all(axis=1).sum()),'reason':'비결측 여부만 확인. 0 또는 비활성 기록을 포함하므로 유효 표본과 구분'},
 {'join':'대표품목 공정4조건 유효','denominator_name':'대표품목 크롬 통과행','denominator':len(representative),'matched':int(active_representative[['density_actual_a_dm2','voltage_clean_v','temperature_c_avg','duration_actual_s']].gt(0).all(axis=1).sum()),'reason':'활성 공정 및 전류·면적·시간·온도 양수, 전압>0.1V 진단 가정'}]
for lag in LAGS:
    dates=valid.date-pd.to_timedelta(lag,unit='D')
    f=features.reindex(dates).reset_index(drop=True)
    for desc,cs in [('전체품목 공정 일자',['density_set_a_dm2','temperature_c_avg','duration_actual_s']),('약품 일자',list(chem_wide.columns)),('도금액 1라인 성분',list(solution_wide.filter(regex='도금조_L1_').columns))]:
        match=f[cs].notna().all(axis=1)
        join_rows.append({'join':f'{desc} 시차{lag}일','denominator_name':'유효 일별 검사 레코드','denominator':len(valid),'matched':int(match.sum()),'reason':'날짜 수준 연결. 개별 제품 노출 또는 LOT 연결 아님'})
    for item in ITEMS:
        cs=[item+'__'+c for c in ['density_set_a_dm2','voltage_clean_v','temperature_c_avg','duration_actual_s']]
        join_rows.append({'join':f'{item} 공정4조건 시차{lag}일','denominator_name':'유효 일별 검사 레코드','denominator':len(valid),'matched':int(f[cs].notna().all(axis=1).sum()),'reason':'해당 품번 공정 존재일과 전체품목 검사일 연결. 해당 품번 불량률 아님'})
joins=pd.DataFrame(join_rows);joins['rate']=joins.matched/joins.denominator
save(joins,'join_coverage');display(joins)
key_audit=pd.DataFrame([
 ['생산작업이력','Part- 번호','있음','대표 3품번 정확 일치'],['생산작업이력','FB- Ident no.','있음','설비 운행 ID. 기업 LOT와 대응표 없음'],['생산작업이력','할당','마스킹','****************'],
 ['일별 검사','품번·LOT·라인','없음','날짜/불량유형/검사수량 전체 합계'],['월별 보고','품명','부분 있음','WORST TOP5 등 선택 표본. 모든 품목 검사분모 없음'],['액분석','라인1/3/4','있음','생산 스테이션14/15/16 대응 미확정'],['약품투입','스테이션13,14~16','있음','검사 LOT는 없음']
],columns=['자료','연결키','상태','판정']);save(key_audit,'key_availability')
missing_target=daily[~daily.total_reviewed.gt(0)].merge(process_daily[['chrome_events']].reset_index(),on='date',how='left')
save(missing_target,'missing_reviewed_days_with_production');display(missing_target[['date','total_reviewed','chrome_events']])

## 8. 0·5·10일 시차 비교
검사일과 그보다 0일·5일·10일 전의 공정기록을 비교합니다.
시차별 가용 날짜와 세 시차에 공통으로 있는 날짜를 각각 계산합니다.
순위상관, 월·요일·검사량을 통제한 상관과 다중비교 보정값을 저장합니다. 상관관계만으로 원인을 판단하지 않습니다.


In [ ]:
def bh(p):
    p=np.asarray(p,float);result=np.full(len(p),np.nan);ok=np.isfinite(p)
    order=np.argsort(p[ok]);vals=p[ok][order];adj=np.minimum.accumulate((vals*len(vals)/np.arange(1,len(vals)+1))[::-1])[::-1]
    pos=np.flatnonzero(ok)[order];result[pos]=np.minimum(adj,1);return result
def residual_corr(x,y,controls):
    xr=stats.rankdata(x);yr=stats.rankdata(y)
    c=np.column_stack([np.ones(len(x)),np.asarray(controls,float)])
    rx=xr-c@np.linalg.lstsq(c,xr,rcond=None)[0];ry=yr-c@np.linalg.lstsq(c,yr,rcond=None)[0]
    return float(np.corrcoef(rx,ry)[0,1]) if np.std(rx)>1e-10 and np.std(ry)>1e-10 else np.nan
lag_matrices={lag:features.reindex(valid.date-pd.to_timedelta(lag,unit='D')).set_axis(valid.index) for lag in LAGS}
targets=['pphv','pinhole_pit_pphv','thickness_under_pphv']
corr_rows=[]
for feature in features.columns:
    common=np.logical_and.reduce([lag_matrices[k][feature].notna().to_numpy() for k in LAGS])
    for lag in LAGS:
        for target in targets:
            for sample in ['available','common_3_lags']:
                mask=lag_matrices[lag][feature].notna()&valid[target].notna()
                if sample=='common_3_lags':mask &= common
                x=lag_matrices[lag].loc[mask,feature];y=valid.loc[mask,target];n=len(x)
                rr=pp=partial=np.nan
                if n>=20 and x.nunique()>1 and y.nunique()>1:
                    rr,pp=stats.spearmanr(x,y)
                    controls=pd.get_dummies(pd.DataFrame({'month':valid.loc[mask,'month'],'weekday':valid.loc[mask,'date'].dt.dayofweek.astype(str)}),drop_first=True,dtype=float)
                    controls['log_reviewed']=np.log1p(valid.loc[mask,'total_reviewed'])
                    controls['log_process_events']=np.log1p(lag_matrices[lag].loc[mask,'chrome_events']).fillna(0)
                    partial=residual_corr(x,y,controls)
                corr_rows.append({'feature':feature,'lag_days':lag,'target':target,'sample':sample,'n_days':n,'spearman_r':rr,'p_nominal':pp,'month_weekday_volume_adjusted_r':partial,'unit_of_analysis':'all-item inspection day'})
correlations=pd.DataFrame(corr_rows)
correlations['q_bh']=correlations.groupby(['target','sample']).p_nominal.transform(bh)
save(correlations,'lag_correlations')
display(correlations.query("target=='pinhole_pit_pphv' and sample=='common_3_lags'").assign(abs_r=lambda d:d.spearman_r.abs()).sort_values('abs_r',ascending=False).head(15))

## 9. 현장 설명과 센서 기록의 관계
도금액 불순물과 전압·전류밀도·처리시간의 날짜 관계를 비교합니다.
전압 관계는 설정 전류밀도·온도·처리시간·월을 통제합니다. 실제 평균전류밀도를 통제한 결과도 따로 저장합니다.
도금액과 실제 사용 설비의 대응이 없어 날짜 수준의 탐색으로 해석합니다.


In [ ]:
sensor=active_chrome.groupby('station').agg(events=('station','size'),low_voltage=('voltage_low_flag','sum'),voltage_raw_median=('voltage_v_avg','median'),voltage_clean_median=('voltage_clean_v','median'),current_median=('current_a_avg','median'),temperature_median=('temperature_c_avg','median')).reset_index()
sensor['low_voltage_fraction']=sensor.low_voltage/sensor.events
save(sensor,'station_sensor_diagnostics');display(sensor)
feedback=[]
panel=active_representative.groupby(['date','item','station'])[process_cols].mean().reset_index()
for (item,station),g in panel.groupby(['item','station']):
    g=g.merge(solution_wide.reset_index(),on='date',how='left',validate='many_to_one')
    for line in [1,3,4]:
        for comp in ['3가크롬','철분','이물질']:
            feature=f'도금조_L{line}_{comp}'
            for outcome in ['voltage_clean_v','density_set_a_dm2','duration_actual_s']:
                needed=[feature,outcome,'temperature_c_avg']
                if outcome=='voltage_clean_v':needed+=['density_set_a_dm2','density_actual_a_dm2','duration_actual_s']
                s=g.dropna(subset=needed)
                rr=partial=partial_actual=np.nan
                if len(s)>=30 and s[feature].nunique()>1 and s[outcome].nunique()>1:
                    rr=stats.spearmanr(s[feature],s[outcome]).statistic
                    controls=pd.get_dummies(s.date.dt.strftime('%Y-%m'),drop_first=True,dtype=float)
                    controls['temperature']=s.temperature_c_avg
                    if outcome=='voltage_clean_v':controls['density']=s.density_set_a_dm2;controls['duration']=s.duration_actual_s
                    partial=residual_corr(s[feature],s[outcome],controls)
                    if outcome=='voltage_clean_v':
                        actual_controls=controls.copy();actual_controls['density']=s.density_actual_a_dm2
                        partial_actual=residual_corr(s[feature],s[outcome],actual_controls)
                feedback.append({'item':item,'station':station,'chemical_line':line,'component':comp,'outcome':outcome,'n_days':len(s),'spearman_r':rr,'controlled_rank_r':partial,'voltage_actual_density_sensitivity_r':partial_actual,'control_density':'set_current/area' if outcome=='voltage_clean_v' else 'not included','exposure_mapping':'unconfirmed; date association only'})
feedback=pd.DataFrame(feedback);save(feedback,'field_feedback_revalidation')
thickness=pd.DataFrame([{'calendar_days':len(daily),'valid_days':len(valid),'positive_days':int((valid.thickness_under_count>0).sum()),'defect_count':float(valid.thickness_under_count.sum()),'representative_thickness_measurements':0,'other_part_thickness_records':21,'thickness_unit_confirmed':False,'part_lot_labeled_samples':0,'interpretation':'다른 6품번 21값·8조건·1일. 단위·측정위치·개체 미확정. 대표품목 두께/LOT 예측·공정능력 불가'}])
save(thickness,'thickness_feasibility');display(thickness)
lab_coverage=lab.groupby(['line','component']).agg(measurements=('value','size'),first_date=('date','min'),last_date=('date','max')).reset_index()
lab_coverage['same_day_inspection_days']=lab.groupby(['line','component']).apply(lambda g:g[g.date.isin(valid.date)].date.nunique(),include_groups=False).values
save(lab_coverage,'additive_lab_coverage')
temp_rows=[]
for _,r in manifest[(manifest.source_group=='plating_solution')&manifest.table_name.eq('액온도-측정자료')].iterrows():
    a=rows_for(ROOT/r.csv_relative_path);line=None;controllers={}
    for i,row in enumerate(a):
        mt=re.search(r'([134])라인',row[0])
        if mt:line=int(mt[1])
        if line is None:continue
        if row[2].startswith('Controller'):controllers[line]=row
        if row[2].startswith('실측'):
            for j,v in enumerate(row):
                if j<len(a[3]) and isdate(a[3][j]) and pd.notna(num(v)):
                    cv=num(controllers.get(line,['']*len(row))[j]);temp_rows.append({'date':pd.Timestamp(a[3][j]).normalize(),'line':line,'tank':row[2],'measured_c':num(v),'controller_c':cv,'measured_minus_controller_c':num(v)-cv})
temperature_compare=pd.DataFrame(temp_rows);save(temperature_compare,'temperature_controller_comparison')

## 9-1. 이미지와 측정값 대조
140개 이미지의 분류표를 사용하고 두께·마이크로크랙·이온교환 수치를 원본 셀과 대조합니다.
두께 21개 값은 다른 6품번의 기록이며 대표품목 실측으로 합산하지 않습니다.
사진의 관찰 길이나 배율을 도금두께로 사용하지 않습니다.


In [ ]:
image_summary=json.loads((IMAGE_AUDIT/'summary.json').read_text(encoding='utf-8'))
expected_inputs=pd.read_csv(IMAGE_AUDIT/'05_재실행도구/analysis_input_sha256.csv')
assert all(digest(_input_file(row.file))==row.sha256 for row in expected_inputs.itertuples()), '고정 입력과 다른 파일: 변환CSV와 이미지검토를 먼저 갱신하세요.'
image_tables={p.stem:pd.read_csv(p) for p in (IMAGE_AUDIT/'03_검토표').glob('*.csv')}
measurement_source=next((ROOT/'raw/도금액분석기록').rglob('*2026년.xlsx'))
wb_measure=load_workbook(measurement_source,read_only=False,data_only=True)
source_checks=[]
for name in ['두께_실측기록','마이크로크랙_시험기록']:
    frame=image_tables[name].copy()
    for row in frame.itertuples():
        ws=wb_measure[row.source_sheet]
        actual=ws[row.source_cell].value
        same=bool(np.isclose(actual,row.value))
        date_value=ws[row.context_date_cell].value
        same=same and pd.Timestamp(date_value).normalize()==pd.Timestamp(row.date)
        same=same and str(ws[row.context_part_cell].value)==str(row.part_number)
        actual_spec=ws[row.spec_cell].value
        same=same and ((actual_spec is None and pd.isna(row.spec_source)) or str(actual_spec)==str(row.spec_source))
        source_checks.append({'source_sheet':row.source_sheet,'source_cell':row.source_cell,'value':actual,'passed':same})
    save(frame,'image_'+name)
ion=image_tables['이온교환_원본셀'].copy()
for i,row in ion.iterrows():
    ws=wb_measure[row.source_sheet]
    for col,field in [('before_cell','before'),('after_cell','after')]:
        actual=num(ws[row[col]].value)
        source_checks.append({'source_sheet':row.source_sheet,'source_cell':row[col],'value':actual,'passed':bool(np.isclose(actual,row[field],equal_nan=True))})
        ion.at[i,field]=actual
ion['computed_efficiency_percent']=100*(ion.before-ion.after)/ion.before.where(ion.before>0)
save(ion,'image_이온교환_원본셀')
wb_measure.close()
image_cell_checks=pd.DataFrame(source_checks);save(image_cell_checks,'image_source_cell_checks')
assert image_cell_checks.passed.all(), '이미지 검토 숫자와 현재 원본 셀 불일치'
for name,frame in image_tables.items():
    if name not in ['두께_실측기록','마이크로크랙_시험기록','이온교환_원본셀']:
        save(frame,'image_'+name)
thickness_records=image_tables['두께_실측기록']
assert len(thickness_records)==21 and not thickness_records.is_representative.any()
assert image_summary['files_checked']==len(_PREPARED_INPUTS['review_inventory']) and image_summary['unique_image_assets']==140
assert len(image_tables['이미지별_분류'])==140
display(thickness_records[['date','part_number','source_sheet','source_cell','value','unit','spec_source']])
display(ion[['date','component','before','after','computed_efficiency_percent','threshold_comparison']])

## 10. 월별 보고서와 검사 집계 비교
월별 보고서의 공정 불량과 출하검사는 집계 범위와 분모가 다릅니다. 원본 표를 추출해 구분하여 비교합니다.


In [ ]:
ns={'a':'http://schemas.openxmlformats.org/drawingml/2006/main'}
ppt_tables,ppt_monthly=[],[]
for p in sorted((ROOT/'raw'/'월별불량현황').rglob('*.pptx')):
    yy,mm=map(int,re.search(r'_(\d{2})년\s*(\d+)월',p.name).groups());yy+=2000
    with zipfile.ZipFile(p) as z:
        for sn in sorted(n for n in z.namelist() if re.match(r'ppt/slides/slide\d+\.xml$',n)):
            e=ET.fromstring(z.read(sn))
            for ti,tbl in enumerate(e.findall('.//a:tbl',ns),1):
                arr=[[''.join(t.text or '' for t in cell.findall('.//a:t',ns)) for cell in row.findall('a:tc',ns)] for row in tbl.findall('a:tr',ns)]
                for ri,row in enumerate(arr):
                    for ci,v in enumerate(row):ppt_tables.append({'source_file':p.relative_to(ROOT).as_posix(),'slide':sn,'table':ti,'row':ri+1,'column':ci+1,'text':v})
                labels={re.sub(r'\s+','',row[0]):row for row in arr if row}
                if '생산수량' not in labels or '불량수량' not in labels or not any('실적(ppm)' in k.lower() for k in labels):continue
                header=next((row for row in arr if f'{mm}월' in [v.strip() for v in row]),None)
                if header is None:continue
                j=[v.strip() for v in header].index(f'{mm}월')
                actual=next(row for k,row in labels.items() if '실적(ppm)' in k.lower())
                ppt_monthly.append({'month':f'{yy}-{mm:02d}','defects_ppt':num(labels['불량수량'][j]),'production_ppt':num(labels['생산수량'][j]),'ppm_reported':num(actual[j]),'source_file':p.relative_to(ROOT).as_posix(),'slide':sn})
save(pd.DataFrame(ppt_tables),'monthly_ppt_tables_long')
ppt_monthly=pd.DataFrame(ppt_monthly).drop_duplicates(['month','source_file','slide'])
ppt_monthly['ppm_recalculated']=ppt_monthly.defects_ppt/ppt_monthly.production_ppt*1e6
save(ppt_monthly,'monthly_ppt_overall')
monthly_comparison=monthly.merge(ppt_monthly,on='month',how='left',validate='one_to_many')
save(monthly_comparison,'inspection_vs_monthly_ppt');display(monthly_comparison[['month','defects','reviewed','total_pphv','defects_ppt','production_ppt','ppm_reported']])

## 11. 시간순 예측 검증
날짜별 핀홀·피트 지표를 대상으로 Ridge와 Random Forest를 비교합니다.
5개 월별 확장 검증 구간으로 모델과 시차를 선택하고 2026년 5월을 최종 시험에 사용합니다.
결측값 대체와 표준화는 각 학습 구간에서만 수행합니다. 이 결과를 제품별 예방 성능으로 해석하지 않습니다.


In [ ]:
model_features=['density_set_a_dm2','voltage_clean_v','temperature_c_avg','duration_actual_s','dose_etch_chromic_automatic_ml','dose_chrome_chromic_automatic_ml','dose_additive_automatic_ml']
common_process=np.logical_and.reduce([lag_matrices[k][['density_set_a_dm2','temperature_c_avg','duration_actual_s']].notna().all(axis=1).to_numpy() for k in LAGS])
folds=pd.read_csv(ROOT/'prepared/five_fold/fold_assignments.csv',parse_dates=['date'])
print('시간순 검증 공통 표본',int(common_process.sum()),'일 / 유효 검사',len(valid),'일')
def new_model(name):
    estimator=Ridge(alpha=10) if name=='Ridge' else RandomForestRegressor(n_estimators=160,max_depth=4,min_samples_leaf=5,random_state=SEED,n_jobs=1)
    return make_pipeline(SimpleImputer(strategy='median',add_indicator=True,keep_empty_features=True),StandardScaler(),estimator)
def metrics(y,pred,w):
    return {'mae_pp':mean_absolute_error(y,pred),'rmse_pp':float(np.sqrt(mean_squared_error(y,pred))),'weighted_mae_pp':float(np.average(np.abs(np.asarray(y)-pred),weights=w)),'r2':r2_score(y,pred) if len(y)>1 else np.nan}
cv_rows,cv_predictions=[],[]
for lag in LAGS:
    X=lag_matrices[lag][model_features]
    for fold in sorted(pd.to_numeric(folds.fold_id,errors='coerce').dropna().unique()):
        fa=folds[folds.fold_id.astype(str).eq(str(int(fold)))]
        tr=valid.index[valid.date.isin(fa[fa.role.eq('train')].date)&common_process]
        va=valid.index[valid.date.isin(fa[fa.role.eq('validation')].date)&common_process]
        if len(tr)<15 or len(va)<3:
            cv_rows.append({'lag_days':lag,'fold':int(fold),'model':'skipped','train_days':len(tr),'validation_days':len(va)});continue
        y=valid.loc[tr,'pinhole_pit_pphv'];yv=valid.loc[va,'pinhole_pit_pphv']
        for name in ['Median baseline','Ridge','Random Forest']:
            if name=='Median baseline':pred=np.repeat(y.median(),len(va))
            else:
                fitted=new_model(name);fitted.fit(X.loc[tr],y);pred=np.clip(fitted.predict(X.loc[va]),0,None)
            cv_rows.append({'lag_days':lag,'fold':int(fold),'model':name,'train_days':len(tr),'validation_days':len(va),**metrics(yv,pred,valid.loc[va,'total_reviewed'])})
            cv_predictions.extend({'lag_days':lag,'fold':int(fold),'model':name,'date':valid.loc[idx,'date'],'actual':float(actual),'predicted':float(pr),'reviewed':float(valid.loc[idx,'total_reviewed'])} for idx,actual,pr in zip(va,yv,pred))
cv=pd.DataFrame(cv_rows);cvp=pd.DataFrame(cv_predictions)
save(cv,'temporal_cv_folds');save(cvp,'temporal_cv_predictions')
cv_summary=[]
for (lag,name),g in cvp.groupby(['lag_days','model']):cv_summary.append({'lag_days':lag,'model':name,'validation_days':len(g),'fold_count':g.fold.nunique(),**metrics(g.actual,g.predicted.to_numpy(),g.reviewed)})
cv_summary=pd.DataFrame(cv_summary).sort_values('mae_pp');save(cv_summary,'temporal_cv_summary');display(cv_summary)
chosen=cv_summary[cv_summary.model.ne('Median baseline')].iloc[0]
chosen_lag=int(chosen.lag_days);chosen_name=chosen.model
tr=valid.index[(valid.date<pd.Timestamp('2026-05-01'))&common_process]
te=valid.index[(valid.date>=pd.Timestamp('2026-05-01'))&common_process]
fitted=new_model(chosen_name);fitted.fit(lag_matrices[chosen_lag].loc[tr,model_features],valid.loc[tr,'pinhole_pit_pphv'])
test_pred=np.clip(fitted.predict(lag_matrices[chosen_lag].loc[te,model_features]),0,None)
baseline_pred=np.repeat(valid.loc[tr,'pinhole_pit_pphv'].median(),len(te))
holdout=pd.DataFrame([{'model':name,'lag_days':chosen_lag,'train_days':len(tr),'test_days':len(te),**metrics(valid.loc[te,'pinhole_pit_pphv'],pred,valid.loc[te,'total_reviewed'])} for name,pred in [(chosen_name,test_pred),('Median baseline',baseline_pred)]])
save(holdout,'holdout_may_metrics');display(holdout)
test_predictions=valid.loc[te,['date','pinhole_pit_pphv','total_reviewed']].copy();test_predictions['prediction']=test_pred;test_predictions['baseline']=baseline_pred
save(test_predictions,'holdout_may_predictions')

## 12. 그래프
월별 검사 결과, 대표품목의 공정조건, 시차별 상관과 예측 결과를 그립니다.


In [ ]:
fig,ax=plt.subplots(figsize=(10.5,4.4))
x=np.arange(len(monthly));ax.plot(x,monthly.total_pphv,'o-',label='전체 불량 PPHV',color='#355C7D');ax.plot(x,monthly.pinhole_pit_pphv,'o-',label='핀홀·피트 PPHV',color='#B6563B')
ax.set_xticks(x,monthly.month,rotation=25);ax.set_ylabel('불량건수 / 검사수량 × 100');ax.set_title('월별 출하검사 결과');ax.legend(frameon=False);ax.grid(axis='y',alpha=.2)
fig.tight_layout();fig.savefig(FIGURES/'01_monthly_defects.png');plt.close(fig)
fig,axes=plt.subplots(4,1,figsize=(11,9.5),sharex=True)
for ax,col,label in zip(axes,['density_set_a_dm2','voltage_clean_v','temperature_c_avg','duration_actual_s'],['설정 전류밀도 (A/dm²)','전압 (V, >0.1)','평균 온도 (℃)','실제 처리시간 (초)']):
    for item,g in item_daily.groupby('item'):ax.plot(g.date,g[col],'.',ms=3,label=item,alpha=.8)
    ax.set_ylabel(label);ax.grid(axis='y',alpha=.2)
axes[0].legend(frameon=False,fontsize=8,ncol=3);axes[0].set_title('대표품목 크롬 공정의 일평균 조건');fig.tight_layout();fig.savefig(FIGURES/'02_representative_conditions.png');plt.close(fig)
heat_features=['density_set_a_dm2','voltage_clean_v','temperature_c_avg','duration_actual_s','dose_etch_chromic_net_ml','dose_chrome_chromic_net_ml','dose_additive_net_ml','도금조_L1_이물질','도금조_L3_이물질','도금조_L4_이물질']
heat=correlations[(correlations.target=='pinhole_pit_pphv')&(correlations['sample']=='common_3_lags')&correlations.feature.isin(heat_features)].pivot(index='feature',columns='lag_days',values='spearman_r').reindex(heat_features)
heatn=correlations[(correlations.target=='pinhole_pit_pphv')&(correlations['sample']=='common_3_lags')&correlations.feature.isin(heat_features)].pivot(index='feature',columns='lag_days',values='n_days').reindex(heat_features)
fig,ax=plt.subplots(figsize=(9.5,6.8));im=ax.imshow(heat.values,cmap='RdBu_r',vmin=-1,vmax=1,aspect='auto')
labels=['설정 전류밀도','전압 (센서 진단 제외)','평균 온도','실제 처리시간','에칭 크롬산 순투입(ml)','도금 크롬산 순투입(ml)','첨가제 순투입(ml)','도금액 1라인 이물질','도금액 3라인 이물질','도금액 4라인 이물질']
ax.set_yticks(range(len(heat)),labels);ax.set_xticks(range(3),[f'{lag}일' for lag in LAGS])
for i in range(len(heat)):
    for j in range(3):
        v=heat.iloc[i,j];ax.text(j,i,f'{v:.2f}\nn={heatn.iloc[i,j]:.0f}' if pd.notna(v) else f'표본 부족\nn={heatn.iloc[i,j]:.0f}',ha='center',va='center',color='white' if abs(v)>.65 else '#17232D',fontsize=9)
ax.set_title('전체품목 핀홀·피트 PPHV와 시차별 순위상관\n성분 라인과 생산설비의 대응 미확정');fig.colorbar(im,ax=ax,label='Spearman r',shrink=.75);fig.tight_layout();fig.savefig(FIGURES/'03_lag_comparison.png');plt.close(fig)
sol_features=[f'도금조_L{line}_{comp}' for line in [1,3,4] for comp in ['크롬산','3가크롬','철분','황산','이물질']]
sa=correlations[(correlations.target=='pinhole_pit_pphv')&(correlations['sample']=='available')&correlations.feature.isin(sol_features)]
sh=sa.pivot(index='feature',columns='lag_days',values='spearman_r').reindex(sol_features)
sn=sa.pivot(index='feature',columns='lag_days',values='n_days').reindex(sol_features)
fig,ax=plt.subplots(figsize=(9,8));im=ax.imshow(sh.values,cmap='RdBu_r',vmin=-1,vmax=1,aspect='auto')
ax.set_yticks(range(len(sh)),[f'{line}라인 {comp}' for line in [1,3,4] for comp in ['크롬산','3가크롬','철분','황산','이물질']]);ax.set_xticks(range(3),[f'{k}일' for k in LAGS])
for i in range(len(sh)):
    for j in range(3):
        v=sh.iloc[i,j];ax.text(j,i,f'{v:.2f} (n={sn.iloc[i,j]:.0f})' if pd.notna(v) else f'n={sn.iloc[i,j]:.0f}',ha='center',va='center',color='white' if abs(v)>.65 else '#17232D',fontsize=9)
ax.set_title('도금액 성분과 전체품목 핀홀·피트 PPHV\n각 시차별 가용 표본. 동일 표본 비교 아님');fig.colorbar(im,ax=ax,label='Spearman r',shrink=.7);fig.tight_layout();fig.savefig(FIGURES/'05_solution_available_lags.png');plt.close(fig)
fig,ax=plt.subplots(figsize=(10.5,4.3));ax.plot(test_predictions.date,test_predictions.pinhole_pit_pphv,'o-',label='실측');ax.plot(test_predictions.date,test_predictions.prediction,'o-',label=f'{chosen_name}, {chosen_lag}일');ax.plot(test_predictions.date,test_predictions.baseline,'--',label='학습 중앙값 기준선')
ax.set_ylabel('핀홀·피트 PPHV');ax.set_title('2026-05 최종 시험: 전체품목 일별 집계');ax.legend(frameon=False);ax.grid(axis='y',alpha=.2);fig.autofmt_xdate();fig.tight_layout();fig.savefig(FIGURES/'04_holdout_predictions.png');plt.close(fig)
for fname in ['01_monthly_defects.png','03_lag_comparison.png']:display(Image(filename=str(FIGURES/fname)))

## 13. 원본 대조와 실행 검증
원본 숫자 셀, 합계, 날짜 중복, 학습·시험 분리와 입력파일 해시를 확인합니다.


In [ ]:
import xlrd
extra_checks=[]
for _,r in manifest[manifest.source_group.isin(['chemical_inputs','power_usage','production_history'])].iterrows():
    source=ROOT/r.source_relative_path;checked=0;errors=0
    if r.source_group=='production_history':
        with (ROOT/r.csv_relative_path).open(encoding='utf-8-sig',newline='') as f:
            reader=csv.reader(f);ar=[next(reader) for _ in range(4)]
        wb=load_workbook(source,read_only=True,data_only=True);ws=wb[r.table_name];rr=list(ws.iter_rows(min_row=1,max_row=4,values_only=True))
        for i in range(1,4):
            for j in [0,1,2,3,7,9,11,13,17,19]:
                value=rr[i][j];csvvalue=ar[i][j];checked+=1
                if isinstance(value,datetime):same=pd.Timestamp(value)==pd.Timestamp(csvvalue)
                elif isinstance(value,(int,float)):same=np.isclose(value,num(csvvalue),equal_nan=True)
                else:same=str(value or '').replace('\r\n','\n')==csvvalue.replace('\r\n','\n')
                errors+=not same
        wb.close()
    else:
        ar=rows_for(ROOT/r.csv_relative_path);wb=xlrd.open_workbook(source,on_demand=True);ws=wb.sheet_by_name(r.table_name)
        numeric_cells=[(i,j) for i,row in enumerate(ar) for j,v in enumerate(row) if pd.notna(num(v))]
        # 각 원본의 처음·중간·마지막 구간에서 숫자 셀을 표본 점검합니다.
        pick=sorted(set(np.linspace(0,len(numeric_cells)-1,min(90,len(numeric_cells)),dtype=int)))
        for k in pick:
            i,j=numeric_cells[k];value=ws.cell_value(i,j);checked+=1
            errors+=not np.isclose(num(value),num(ar[i][j]),equal_nan=True)
        wb.release_resources()
    extra_checks.append({'source_file':r.source_relative_path,'cells_checked':checked,'mismatches':errors})
all_raw_checks=pd.concat([pd.DataFrame(raw_checks),pd.DataFrame(extra_checks)],ignore_index=True);save(all_raw_checks,'original_cell_checks')
solution_checks=[]
for (source,sheet),g in solution_all.groupby(['source_file','source_sheet']):
    selected=g[g.value_raw.notna()].iloc[np.linspace(0,g.value_raw.notna().sum()-1,min(100,g.value_raw.notna().sum()),dtype=int)]
    wb=load_workbook(ROOT/source,read_only=True,data_only=True);ws=wb[sheet]
    rr=list(ws.iter_rows(min_row=1,max_row=int(selected.source_row.max()),values_only=True));errors=0
    for _,row in selected.iterrows():errors+=not np.isclose(num(rr[int(row.source_row)-1][int(row.source_col)-1]),row.value_raw,equal_nan=True)
    solution_checks.append({'source_file':source,'cells_checked':len(selected),'mismatches':errors});wb.close()
all_raw_checks=pd.concat([all_raw_checks,pd.DataFrame(solution_checks)],ignore_index=True);save(all_raw_checks,'original_cell_checks')
verification=[
 ['검사 날짜 중복 없음',not daily.date.duplicated().any(),f'{len(daily)} 날짜'],
 ['206일 표본 재현',len(valid)==206,'검사수량>0'],
 ['유효 206일 기존 불량·검사수량 일치',bool((repro.loc[repro.valid_denominator,['count_difference','reviewed_difference']]==0).all().all()),'원본 유효 분모 날짜만 비교'],
 ['기존 0 대체와 원본 빈칸 구분',int(repro.prepared_zero_for_source_blank.sum())==6,'검사수량 빈칸6일을 결측 유지'],
 ['불량유형 합계=Total',bool(np.allclose(daily.total_defects,daily.defect_type_sum,equal_nan=False)),'26개 유형'],
 ['원본 대조',bool(all_raw_checks.mismatches.eq(0).all()),f'{int(all_raw_checks.cells_checked.sum())} 셀; {all_raw_checks.source_file.nunique()} 파일'],
 ['약품 날짜·종류 중복 없음',not chem.duplicated(['date','chemical']).any(),f'{len(chem)} 일자×약품'],
 ['약품 월계=일계(0.1ml 허용)',bool(chem_audit.monthly_minus_daily.abs().max()<.1),f'최대차 {chem_audit.monthly_minus_daily.abs().max():.6f}ml'],
 ['생산 공정행 중복 제거',not chrome.duplicated(dupekeys).any(),f'제거 {int(duplicate_mask.sum())}행'],
 ['상위 사본과 raw 동일',bool(pd.DataFrame(duplicates).hash_equal.all()),f'{len(duplicates)} 사본'],
 ['시차 방향',all((valid.date-(valid.date-pd.to_timedelta(k,unit='D'))).dt.days.eq(k).all() for k in LAGS),'검사일 t, 공정일 t-lag'],
 ['최종 시험 분리',valid.loc[tr,'date'].max()<valid.loc[te,'date'].min(),'5월 시험, 이전월 학습'],
 ['공통 표본 시간순 5개 검증',cvp.fold.nunique()==5,f'{cvp.fold.nunique()}개'],
]
unchanged_original=all(digest(ROOT/path)==h for path,h in initial_hashes.items())
unchanged_intermediate=all(digest(ROOT/path)==h for path,h in input_fingerprints.items())
verification += [['원본 해시 보존',unchanged_original,f'{len(initial_hashes)} 파일'],['기존 CSV/prepared/지시문 해시 보존',unchanged_intermediate,f'{len(input_fingerprints)} 파일']]
verification=pd.DataFrame(verification,columns=['check','passed','detail']);save(verification,'verification')
display(verification)
assert verification.passed.all(), '검증 실패는 verification.csv에서 확인하세요. 결과 해석 전에 해소해야 합니다.'

## 14. 분석 결과 저장
계산표, 그래프, 실행 환경과 분석 보고서를 결과 폴더에 저장합니다.


In [ ]:
total_defects = float(valid.total_defects.sum())
reviewed_total = float(valid.total_reviewed.sum())
pinhole_total = float(valid.pinhole_pit_count.sum())
headline = {
    'inspection_calendar_days': len(daily),
    'inspection_valid_days': len(valid),
    'first_inspection_date': str(valid.date.min().date()),
    'last_inspection_date': str(valid.date.max().date()),
    'total_defects': total_defects,
    'total_reviewed': reviewed_total,
    'weighted_total_pphv': 100 * total_defects / reviewed_total,
    'pinhole_pit_count': pinhole_total,
    'weighted_pinhole_pit_pphv': 100 * pinhole_total / reviewed_total,
    'pinhole_pit_share_of_defects': pinhole_total / total_defects,
    'production_raw_rows': int(pd.DataFrame(production_audit).data_rows.sum()),
    'production_station_present_rows': int(pd.DataFrame(production_audit).station_present_rows.sum()),
    'production_entirely_blank_rows': int(pd.DataFrame(production_audit).entirely_blank_rows.sum()),
    'production_station_missing_nonempty_rows': int(pd.DataFrame(production_audit).station_missing_nonempty_rows.sum()),
    'representative_four_conditions_present': 12902,
    'representative_four_conditions_valid': int(joins.loc[joins['join'].eq('대표품목 공정4조건 유효'), 'matched'].iloc[0]),
    'thickness_records_other_parts': len(thickness_records),
    'representative_thickness_records': int(thickness_records.is_representative.sum()),
    'unique_image_assets': len(image_tables['이미지별_분류']),
    'chrome_events': len(chrome),
    'active_chrome_events': len(active_chrome),
    'representative_events': len(representative),
    'strict_lot_linked_records': 0,
    'model_common_days': int(common_process.sum()),
    'chosen_lag': chosen_lag,
    'chosen_model': chosen_name,
    'original_files': len(inventory),
    'source_conflict_keys': len(conflict_keys),
}
(OUT / 'summary.json').write_text(json.dumps(headline, ensure_ascii=False, indent=2), encoding='utf-8')

coverage=pd.DataFrame([
 ['생산 변환 범위',14,int(pd.DataFrame(production_audit).data_rows.sum()),str(pd.DataFrame(production_audit).first_valid_end_time.min().date()),str(pd.DataFrame(production_audit).last_valid_end_time.max().date()),'설비명 없는 행·빈 서식행 포함. 유효 공정 또는 독립 제품 수가 아님'],
 ['설비명이 있는 생산기록',14,int(pd.DataFrame(production_audit).station_present_rows.sum()),str(chrome.date.min().date()),str(chrome.date.max().date()),'설비명이 있는 기록행. 기간은 크롬공정 관측 기준. 재통과 포함 가능'],
 ['크롬 공정',14,len(chrome),str(chrome.date.min().date()),str(chrome.date.max().date()),'0전류/면적/처리시간은 활성 분석에서 제외'],
 ['대표품목 크롬 공정',14,len(representative),str(representative.date.min().date()),str(representative.date.max().date()),'3개 정확한 품번'],
 ['출하검사 일자',7,len(daily),str(daily.date.min().date()),str(daily.date.max().date()),f'유효 분모 {len(valid)}일'],
 ['약품 일계',42,len(chem),str(chem.date.min().date()),str(chem.date.max().date()),'일자×약품3종'],
 ['액 성분 유효 측정',2,int(solution.value.notna().sum()),str(solution[solution.value.notna()].date.min().date()),str(solution[solution.value.notna()].date.max().date()),'일자×라인×액조×성분'],
 ['기타품목 두께 기록',1,len(thickness_records),'2026-06-22','2026-06-22','6품번·8조건·21숫자. 독립개체·단위 미확정, 대표품번0'],
 ['마이크로크랙 숫자',1,len(image_tables['마이크로크랙_시험기록']),str(image_tables['마이크로크랙_시험기록'].date.min()),str(image_tables['마이크로크랙_시험기록'].date.max()),'52숫자. 측정법·단위·반복 미확정'],
 ['이온교환 원본 셀',1,len(ion),str(ion.date.min()),str(ion.date.max()),'16시험조건×4성분. 전후값완비14조건·대상정화율28값'],
 ['첨가제 농도',1,len(lab),str(lab.date.min().date()),str(lab.date.max().date()),'%rel, 불규칙 측정. 일별 보간 없음'],
 ['전력 월계',14,len(power),power.month.min(),power.month.max(),'월×스테이션'],
 ['월별 보고서',15,len(ppt_monthly),ppt_monthly.month.min(),ppt_monthly.month.max(),'출하검사와 모집단 다름'],
],columns=['자료','파일 수','분석 레코드','시작','종료','레코드 정의']);save(coverage,'data_coverage')
quality=pd.DataFrame([
 ['두께 실측기록의 범위','다른 6품번21값·8조건·1일, 대표0값','단위·반복/개체 미확정으로 공정능력/대표품목 예측은 보류','image_두께_실측기록.csv'],
 ['공정 4조건의 유효 표본','값 존재12,902 / 유효12,864','38행을 활성·양수 조건에서 추가 제외','join_coverage.csv'],
 ['이온교환 이미지 충돌','이미지에만3조건·날짜충돌1조건','연도 추정·날짜 충돌·표시정밀도 별도 보존','image_이온교환_이미지대조.csv'],
 ['검사수량 빈칸',f'{len(daily)-len(valid)}일','기존 prepared는0으로 대체. 결측 유지 및 생산로그 대조','missing_reviewed_days_with_production.csv'],
 ['5월 검사 파일의 4월1일 열','2,979 불량 / 22,951 검사','5월 재집계에서 제외. source Total과 달라짐','inspection_sheet_audit.csv'],
 ['검사 시트 수식 오류',str(sum(x['formula_error_cells'] for x in daily_audit))+'셀','원본 수정 없이 날짜/유형별 숫자 다시 집계','inspection_sheet_audit.csv'],
 ['16번 스테이션 저전압',str(int(sensor[sensor.station.str.startswith('16')].low_voltage.sum()))+'행','<=0.1V를 진단상 제외. 원자료 전압과 대조','station_sensor_diagnostics.csv'],
 ['액 성분 시트 충돌',str(len(conflict_keys))+'키','우선순위 적용, 충돌 원자료 보존','solution_conflicting_sources.csv'],
 ['약품 수동값 음수',str(int(chem_audit.negative_manual_days.sum()))+'일×약품','음수 유지. 순투입과 자동투입을 각각 분석','chemical_reconciliation.csv'],
 ['전류밀도 이미지 산술 불일치',_PREPARED_INPUTS['parameters']['density_example'],'공식에 따라 설정 및 실제평균 밀도 별도 산출','requirements.csv'],
 ['16번 전력계의 저전압 영향','월별 kWh/Ah가 다른 스테이션보다 매우 작음','약품/kWh 비율은 확인용이며 센서 점검 전 효율기준 확정 불가','power_monthly_stations.csv'],
 ['7월30일까지 파일 종료시각',str(chrome.date.max().date()),'자정 통과 이력은 실제 종료일 유지','representative_chrome_events.csv'],
],columns=['점검 항목','규모','처리','근거']);save(quality,'data_quality_issues')
versions={name:__import__(name).__version__ for name in ['pandas','numpy','scipy','sklearn','matplotlib','openpyxl','xlrd']}
(OUT/'environment.json').write_text(json.dumps({'python':sys.version,'platform':platform.platform(),'packages':versions,'seed':SEED,'lags':LAGS,'voltage_floor':VOLTAGE_FLOOR},ensure_ascii=False,indent=2),encoding='utf-8')

def table_html(df,limit=None):
    if limit:df=df.head(limit)
    labels={'month':'월','calendar_days':'날짜 수','valid_days':'유효 일수','defects':'불량건수','reviewed':'검사수량','pinhole_pit':'핀홀·피트 건수','thickness_under':'두께미달 건수','total_pphv':'전체 PPHV(%)','pinhole_pit_pphv':'핀홀·피트 PPHV(%)','join':'연결 정의','denominator_name':'분모 정의','denominator':'분모 수','matched':'연결 수','rate':'연결 비율','reason':'한계','item':'품번','events':'공정 통과행','fb_ids':'설비 운행ID 수','first_date':'시작일','last_date':'종료일','days':'공정 존재일','logged_parts':'기록 수량 합계','low_voltage_events':'저전압 기록','station':'스테이션','low_voltage':'저전압 기록','voltage_raw_median':'원전압 중앙값(V)','voltage_clean_median':'진단 제외 후 전압(V)','current_median':'전류 중앙값(A)','temperature_median':'온도 중앙값(℃)','low_voltage_fraction':'저전압 비율','feature':'변수','lag_days':'시차(일)','n_days':'분석 일수','spearman_r':'순위상관 r','month_weekday_volume_adjusted_r':'월·요일·물량 통제 r','q_bh':'BH 보정 q','line':'라인','bath':'액조','component':'성분','measured':'실측 수','spec_evaluable':'규격 판정 수','spec_exceed':'규격 이탈 수','management_evaluable':'관리치 판정 수','management_exceed':'관리치 이탈 수','measurements':'실측 수','same_day_inspection_days':'검사일과 겹치는 날','chemical':'약품 구분','daily_rows':'일계 수','monthly_minus_daily':'월계-일계(ml)','daily_minus_events':'일계-투입내역(ml)','model':'모형','validation_days':'검증 일수','fold_count':'검증 월 수','mae_pp':'평균 절대오차(%p)','rmse_pp':'평균 제곱근오차(%p)','weighted_mae_pp':'검사량 가중오차(%p)','r2':'결정계수 R²','train_days':'학습 일수','test_days':'시험 일수','defects_ppt':'PPT 공정불량 수','production_ppt':'PPT 생산수량','ppm_reported':'PPT 실적(ppm)','check':'검증 항목','passed':'통과','detail':'내용','title':'원본 제목','date_columns':'해당월 날짜 열','excluded_other_month_dates':'제외한 다른달 날짜','sheet_total':'원시트 불량 월계','date_column_total':'해당월 불량 재집계','sheet_reviewed':'원시트 검사 월계','date_column_reviewed':'해당월 검사 재집계','formula_error_cells':'수식 오류 셀'}
    labels.update({'events':'이력 건수','missing_reviewed_dates':'검사수량 결측 일수'})
    labels.update({'net_ml_per_kwh__etch_chromic':'에칭 크롬산 순투입(ml/kWh)','net_ml_per_kwh__chrome_chromic':'도금 크롬산 순투입(ml/kWh)','automatic_ml_per_kwh__etch_chromic':'에칭 크롬산 자동투입(ml/kWh)','automatic_ml_per_kwh__chrome_chromic':'도금 크롬산 자동투입(ml/kWh)','additive_net_ml':'첨가제 순투입(ml)','additive_automatic_ml':'첨가제 자동투입(ml)'})
    formatted=df.rename(columns=labels)
    formatters={'BH 보정 q':lambda x:'<0.001' if x<0.001 else f'{x:.3f}'} if 'BH 보정 q' in formatted else None
    content=formatted.to_html(index=False,escape=True,border=0,na_rep='결측/미확인',float_format=lambda x:f'{x:,.3f}',formatters=formatters)
    content=re.sub(r'([A-Za-z0-9_]+\.csv)',lambda m:f'<a href="tables/{m[1]}">{m[1]}</a>',content)
    return '<div class="table-wrap">'+content+'</div>'
def img_html(name):
    b=base64.b64encode((FIGURES/name).read_bytes()).decode()
    return f'<img class="chart" src="data:image/png;base64,{b}" alt="{html.escape(name)}">'
global_lag=correlations[(correlations.target=='pinhole_pit_pphv')&(correlations['sample']=='common_3_lags')&correlations.feature.isin(heat_features)][['feature','lag_days','n_days','spearman_r','month_weekday_volume_adjusted_r','q_bh']]
item_lag=correlations[(correlations.target=='pinhole_pit_pphv')&(correlations['sample']=='common_3_lags')&correlations.feature.str.contains('__')].copy()
item_lag['abs_r']=item_lag.spearman_r.abs();item_lag=item_lag.sort_values('abs_r',ascending=False)
best_hold=holdout[holdout.model.ne('Median baseline')].iloc[0];base_hold=holdout[holdout.model.eq('Median baseline')].iloc[0]
performance_note='시험 MAE가 중앙값 기준선보다 낮았습니다. 단일 시험월 결과이며 운영 성능으로 확정하지 않습니다.' if best_hold.mae_pp<base_hold.mae_pp else '시험 MAE가 중앙값 기준선보다 높았습니다. 현 상태에서 예측모형을 운영 기준으로 권고하지 않습니다.'


chemical_gaps = []
chemical_calendar = pd.date_range(chem.date.min(), chem.date.max())
for kind, group in chem.groupby('chemical'):
    for date in chemical_calendar.difference(group.date):
        chemical_gaps.append({'date': date.strftime('%Y-%m-%d'), 'chemical': kind,
                              'status': '일계 행 미기록; 휴무·0투입·누락 미확정'})
save(pd.DataFrame(chemical_gaps), 'chemical_missing_calendar_days')

sections = [
    ('주요 결과', f'''<p>검사수량이 있는 {len(valid)}일을 기준으로 계산했습니다.</p>
<ul><li>검사수량 {reviewed_total:,.0f}개, 불량기록 {total_defects:,.0f}건</li>
<li>핀홀·피트 {pinhole_total:,.0f}건, 전체 불량기록 중 {100*pinhole_total/total_defects:.1f}%</li>
<li>대표품목 크롬 공정기록 {len(representative):,}행</li></ul>
<p>불량기록은 같은 제품에 중복 집계될 수 있습니다. 불량기록 비율을 고유 불량품 비율로 해석하지 않습니다.</p>''' + img_html('01_monthly_defects.png')),
    ('분석대상과 계산 기준', table_html(requirements) + table_html(theme_reference)
     + '<p>테마 문서의 예시 표는 설명용이며 실제 측정값에 포함하지 않습니다.</p>'),
    ('입력자료와 표본 수', table_html(coverage)
     + f'<p>raw의 원본은 {len(inventory)}개입니다. 같은 내용의 상위 폴더 사본 {len(duplicates)}개는 중복 집계하지 않습니다.</p>'),
    ('검사 집계', table_html(monthly)
     + '<p>검사수량이 빈 날짜는 결측으로 유지합니다. 월 비율은 월 불량건수 합계를 월 검사수량 합계로 나누어 계산합니다.</p>'),
    ('생산·검사 기록의 연결 범위', table_html(joins) + table_html(key_audit)
     + '<p>검사자료에는 품번과 LOT가 없습니다. 날짜 연결은 날짜 수준의 비교이며, LOT 연결률의 분모는 확인할 수 없습니다.</p>'),
    ('대표품목 공정조건', table_html(item_summary) + img_html('02_representative_conditions.png')
     + table_html(sensor)
     + '<p>공정 통과행은 제품 개수와 다릅니다. 저전압 기록은 원값과 진단용 제외값을 구분해 보관합니다.</p>'),
    ('0·5·10일 시차 비교', img_html('03_lag_comparison.png') + table_html(global_lag)
     + '<p>검사일에서 0일, 5일, 10일을 뺀 공정일을 연결했습니다. 상관관계는 원인이나 권장 작업조건을 뜻하지 않습니다.</p>'),
    ('도금액 성분과 약품 기록', table_html(spec_summary[spec_summary.bath.eq('도금조')])
     + table_html(lab_coverage) + img_html('05_solution_available_lags.png')
     + '<p>도금액 성분은 측정된 날짜만 사용했습니다. 약품 순투입량에는 음수 수동 정정값을 유지했으며 자동투입량도 따로 계산했습니다.</p>'),
    ('월별 약품량과 전력량', table_html(theme2_monthly)
     + '<p>월별 약품량을 kWh와 Ah로 각각 나누어 비교했습니다. Nr.6는 기록에서 구분되는 첨가제 한 계열입니다. 이 비율만으로 적정 투입량이나 절감 효과를 확정하지 않습니다.</p>'),
    ('센서와 현장 설명에 대한 탐색', table_html(feedback, 18)
     + '<p>품목·설비별 일평균과 라인별 성분값의 날짜 관계입니다. 실제 사용 액조가 연결되지 않아 특정 성분의 영향을 확정할 수 없습니다.</p>'),
    ('시간순 예측 검증', table_html(cv_summary) + table_html(holdout)
     + '<p>Ridge와 Random Forest를 시간순으로 비교했습니다. 최종 시험은 2026년 5월입니다. 전체품목의 날짜별 지표를 대상으로 한 진단이며 제품별 예방 성능은 아닙니다.</p>'
     + img_html('04_holdout_predictions.png')),
    ('월별 보고서와 검사 집계의 차이', table_html(monthly_comparison[
        ['month','defects','reviewed','total_pphv','defects_ppt','production_ppt','ppm_reported']])
     + '<p>월별 보고서와 출하검사는 집계 범위와 분모가 다르므로 합산하거나 대체하지 않습니다.</p>'),
    ('자료 품질과 원본 대조', table_html(quality) + table_html(verification)),
    ('결과 파일', '<p>세부 계산표는 tables, 그래프는 figures 폴더에 있습니다. 입력파일 해시는 tables/source_files_sha256.csv, 실행 환경은 environment.json에 저장합니다.</p>'),
]
css = '''body{font-family:"Malgun Gothic",Arial,sans-serif;color:#222;background:#f5f6f7;margin:0;line-height:1.7}
main{max-width:1120px;margin:28px auto;padding:36px;background:white}h1{font-size:30px}h2{font-size:22px;margin-top:40px;border-bottom:1px solid #ddd;padding-bottom:8px}
p,li{font-size:15px}.table-wrap{overflow-x:auto;margin:18px 0}table{border-collapse:collapse;width:100%;font-size:12px}th,td{padding:9px;border-bottom:1px solid #ddd;text-align:left;vertical-align:top}
th{background:#355c7d;color:white}.chart{max-width:100%;height:auto}a{color:#355c7d}nav{display:flex;gap:8px 18px;flex-wrap:wrap}'''
nav = '<nav>' + ''.join(f'<a href="#s{i}">{html.escape(title)}</a>' for i,(title,_) in enumerate(sections)) + '</nav>'
report = ('<!doctype html><html lang="ko"><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width, initial-scale=1">'
          '<title>한국로텍 데이터 분석 결과</title><style>' + css + '</style></head><body><main><h1>한국로텍 데이터 분석 결과</h1>'
          + f'<p>출하검사 기간: {valid.date.min():%Y-%m-%d}~{valid.date.max():%Y-%m-%d}</p>' + nav
          + ''.join(f'<section id="s{i}"><h2>{html.escape(title)}</h2>{content}</section>' for i,(title,content) in enumerate(sections))
          + '</main></body></html>')
(OUT / '한국로텍_분석보고서.html').write_text(report, encoding='utf-8')
shutil.copy2(_nb_file, OUT / '한국로텍_데이터분석.ipynb')
_output_manifest = TABLES / 'output_manifest.csv'
pd.DataFrame([{'file':p.relative_to(OUT).as_posix(),'bytes':p.stat().st_size}
              for p in sorted(OUT.rglob('*')) if p.is_file() and p != _output_manifest
              and not p.name.startswith('~$')]).to_csv(_output_manifest, index=False, encoding='utf-8-sig')
display(Markdown('**분석 완료:** ' + str(OUT / '한국로텍_분석보고서.html')))
